<p class="lead">This notebook is the R edition of the third unit of the course. The lecture slides are a map; this text is the territory. Unit 2 introduced learning from data and a first classifier, logistic regression, which draws a straight boundary through feature space. This unit presents three further families of classifiers that approach the same problem from three directions: <em>probabilistic</em> methods, which model how the data of each class are generated and then invert the model with Bayes' theorem (naive Bayes, Bayesian networks); <em>geometric</em> methods, which search directly for the boundary with the largest margin of safety (support vector machines); and <em>memory-based</em> methods, which store the training sample and classify by analogy with the nearest stored cases (k-nearest neighbours). Every method is first derived, then implemented by hand in a few lines of R, then compared with a package implementation, and finally exercised on the breast-cancer data that recur throughout the course. Section 8 compares the three families honestly, with the methodology of Supplement S1.</p>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>State Bayes' theorem, apply it by hand to a diagnostic test, and explain why a positive result of an accurate test for a rare disease is usually a false positive.</li>
<li>Define the posterior probability, the Bayes classifier and the Bayes error, and show how unequal misclassification costs move the decision threshold.</li>
<li>Derive the Gaussian naive Bayes classifier from the conditional-independence assumption, implement it in base R, and explain why it classifies well even when the assumption is false.</li>
<li>Read a Bayesian network as a factorisation of a joint distribution, perform inference by enumeration on a small medical network, and explain how evidence and d-separation change posteriors.</li>
<li>Describe the k-nearest-neighbours rule, its distance metrics, its dependence on feature scaling and on <em>k</em>, its computational cost, and the curse of dimensionality.</li>
<li>Explain the maximum-margin principle, the roles of support vectors, the parameter <em>C</em> and the kernel, and the relation between the hinge loss and the logistic loss of Unit 2.</li>
<li>Compare the three families on the same data with stratified cross-validation and choose among them from their assumptions, costs and interpretability.</li>
</ul></div>

<div class="box bridge"><h4>How to read this notebook</h4>
<p>Each code chunk is short and commented. Run it, read its output, then read the paragraph that interprets the output. Exercises (green boxes) ask you to modify the chunk above them and re-run; they are the fastest way to learn. Blue boxes carry key ideas, orange boxes carry common misconceptions, and purple boxes connect this unit to other units or to the bioinformatics course (BGM324). The notebook uses only base R together with <code>mlbench</code>, <code>e1071</code>, <code>class</code>, <code>MASS</code> and <code>pROC</code>. In a fresh Google Colab R runtime a student runs once <code>install.packages(c("e1071","randomForest","rpart.plot","neuralnet","cluster","mlbench","pROC"))</code>; no data is downloaded. Four algorithms are traced by hand on paper-sized examples (Sections 2.2, 4.2–4.3, 5.2 and 6.1) before the code confirms them. Evaluation methodology — why cases are held out, what cross-validation is, what the AUC measures — is assumed from Supplement S1 and not re-derived here.</p></div>

<div class="box warn"><h4>Where the R edition differs from the Python course</h4><p>The Python course uses scikit-learn's <code>load_breast_cancer</code>: 569 aspirates and 30 continuous nuclear features. The R mirror uses the Wisconsin data shipped by <code>mlbench</code>,<sup class="c"><a href="#r4">[4]</a>,<a href="#r40">[40]</a></sup> which is the <em>original</em> Wisconsin set: 683 complete cases described by nine features scored as integers from 1 to 10. The methods, the reasoning and the conclusions are identical; only the number and type of features differ, and where the Python text says "30 features" this edition says "nine". The regression demonstrations use <code>BostonHousing</code> (a continuous target) in place of scikit-learn's diabetes data, and the digit demonstrations use small <em>simulated</em> image matrices because R has no built-in digits set. Each substitution is flagged where it occurs.</p></div>



In [ ]:
# Libraries used throughout. Seeds are fixed so every number quoted in the text is reproducible.
suppressPackageStartupMessages({
  library(mlbench); library(e1071); library(class); library(MASS); library(pROC)
})
set.seed(0)
BLUE <- "#2b5d9b"; ORANGE <- "#d2691e"; GREEN <- "#3a7d44"; GREY <- "#7f7f7f"  # benign, malignant, accent, neutral

# Small generators that reproduce scikit-learn's make_blobs / make_moons / make_circles demonstrations.
make_blobs <- function(n, centers, sd = 0.6) {
  k <- nrow(centers); per <- n %/% k
  X <- do.call(rbind, lapply(seq_len(k), function(i) MASS::mvrnorm(per, centers[i, ], diag(sd^2, 2))))
  list(X = X, y = factor(rep(c("neg", "pos"), length.out = k)[rep(seq_len(k), each = per)]))
}
make_moons <- function(n = 300, noise = 0.25) {
  n1 <- n %/% 2; n2 <- n - n1; t1 <- seq(0, pi, length.out = n1); t2 <- seq(0, pi, length.out = n2)
  X <- rbind(cbind(cos(t1), sin(t1)), cbind(1 - cos(t2), 0.5 - sin(t2))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
  list(X = X, y = factor(rep(c("A", "B"), c(n1, n2))))
}
make_circles <- function(n = 300, noise = 0.1, factor = 0.4) {
  n1 <- n %/% 2; n2 <- n - n1; to <- seq(0, 2 * pi, length.out = n1); ti <- seq(0, 2 * pi, length.out = n2)
  X <- rbind(cbind(cos(to), sin(to)), cbind(factor * cos(ti), factor * sin(ti))) + matrix(rnorm(2 * n, 0, noise), ncol = 2)
  list(X = X, y = factor(rep(c("outer", "inner"), c(n1, n2))))
}

# Shade the decision regions of any 2-feature classifier given a function that labels a grid as 0/1 (1 = positive).
shade_regions <- function(predfun, xlim, ylim, cn = c("x", "y"), res = 160, main = "", xlab = cn[1], ylab = cn[2]) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res)
  G <- setNames(expand.grid(gx, gy), cn)
  Z <- matrix(as.integer(predfun(G)), nrow = res)
  image(gx, gy, Z, col = c("#dfe7f1", "#fbe3d3"), main = main, xlab = xlab, ylab = ylab)
}
cat("R", R.version.string, "| e1071", as.character(packageVersion("e1071")),
    "| class", as.character(packageVersion("class")), "| mlbench", as.character(packageVersion("mlbench")), "\n")



The printout records the versions of R<sup class="c"><a href="#r36">[36]</a></sup> and of the packages these notes were built with, as the reproducibility habit of Supplement S0 requires.

## 1. Three ways to draw a boundary

Unit 1 defined a classifier as a rule that partitions feature space into regions and assigns a class to each, and Unit 2 built the first such rule, logistic regression, whose regions are separated by a straight line. There are many ways to arrive at a boundary, and the methods of this unit represent three of them.

1. **Probabilistic (generative) methods** ask how the data of each class are distributed — "what do the features of malignant tumours look like, and of benign ones?" — estimate those distributions from the training sample, and then use Bayes' theorem to turn them into the probability of each class given the features. The boundary is the set of points where two classes are equally probable. Naive Bayes (Section 4) and Bayesian networks (Section 5) belong here.
2. **Geometric (discriminative) methods** do not model the classes at all. They search directly for the boundary that separates the training cases, and among all boundaries that do so, they choose the one that leaves the widest empty corridor — the <em>margin</em> — between the classes. The support vector machine (Section 7) is the canonical example. Logistic regression is also discriminative, but it chooses its line by a different criterion.
3. **Memory-based (instance-based) methods** build no model and no explicit boundary. They store the training sample and, when a new case arrives, look up the most similar stored cases and copy their label. The k-nearest-neighbours rule (Section 6) is the archetype; its boundary exists only implicitly, as the places where the majority among the neighbours changes.

The distinction matters because the three families fail differently. A generative model is only as good as its assumptions about the class distributions; a margin-based model is only as good as its choice of boundary shape; a memory-based model is only as good as its notion of similarity and the density of stored cases. The same two-feature problem will be solved by all three so that the differences can be seen. The data are, as in every unit, the Wisconsin breast-cancer sample.<sup class="c"><a href="#r4">[4]</a></sup> For the two-dimensional pictures we keep the two features used by hand in Unit 1, <em>clump thickness</em> and <em>cell size</em>; the full nine-feature problem returns in Section 8. An introduction to all these methods in R, at the level of this course, is the textbook of James and colleagues.<sup class="c"><a href="#r39">[39]</a></sup>



In [ ]:
# The recurring example: two features of the breast-cancer data, split once into 70% training / 30% test.
data(BreastCancer, package = "mlbench")
bc <- BreastCancer[, -1]                                  # drop the Id column
bc <- bc[complete.cases(bc), ]                            # drop the 16 incomplete records
bc[, 1:9] <- lapply(bc[, 1:9], function(z) as.numeric(as.character(z)))   # ordinal factor -> numeric 1..10
bc$Class <- factor(bc$Class, levels = c("benign", "malignant"))
feat2 <- c("Cl.thickness", "Cell.size")
n <- nrow(bc); set.seed(0); idx <- sample(n, round(0.7 * n))
tr <- bc[idx, ]; te <- bc[-idx, ]
cat(sprintf("training cases: %d (benign %d, malignant %d) | test cases: %d\n",
            nrow(tr), sum(tr$Class == "benign"), sum(tr$Class == "malignant"), nrow(te)))
pcol <- function(d) ifelse(d$Class == "malignant", ORANGE, BLUE)
plot(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.7,
     xlab = "clump thickness (1-10)", ylab = "cell size (1-10)",
     main = "The two-feature problem that all three families will solve")
legend("topleft", c("malignant", "benign"), col = c(ORANGE, BLUE), pch = 19, bty = "n")



<p class="caption"><b>Figure 1.</b> The 478 training cases in the plane of clump thickness and cell size (jittered, because both features are integer scores from 1 to 10). The classes are separable in the bulk — benign tumours cluster at low scores — but overlap in a band of intermediate values; no boundary of any shape will classify every case correctly, and the question for every method is where, and with what shape, to draw the line through the overlap.</p>

The chunk below shows the answer each of the three families gives, with default settings, before any of them has been explained. The figure is a preview to return to: by the end of the unit every feature of the three panels — the curved boundary of naive Bayes, the straight band of the linear support vector machine, the jagged islands of k-nearest neighbours — will have an explanation.



In [ ]:
# Preview: the same problem solved by a probabilistic, a geometric and a memory-based classifier.
f2 <- as.formula(paste("Class ~", paste(feat2, collapse = " + ")))
nb_p  <- naiveBayes(f2, data = tr)
svm_p <- svm(f2, data = tr, kernel = "linear", cost = 1)
ctr <- colMeans(tr[feat2]); csd <- apply(tr[feat2], 2, sd)               # scale for k-NN
sc <- function(df) scale(df[feat2], center = ctr, scale = csd)
xlim <- c(0.5, 10.5); ylim <- c(0.5, 10.5)
preds <- list(
  "Probabilistic: Gaussian naive Bayes" = function(G) predict(nb_p, G) == "malignant",
  "Geometric: linear SVM"               = function(G) predict(svm_p, G) == "malignant",
  "Memory-based: 5 nearest neighbours"  = function(G) knn(sc(tr), scale(G, center = ctr, scale = csd), tr$Class, k = 5) == "malignant")
accs <- c(mean(predict(nb_p, te) == te$Class), mean(predict(svm_p, te) == te$Class),
          mean(knn(sc(tr), sc(te), tr$Class, k = 5) == te$Class))
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (i in seq_along(preds)) {
  shade_regions(preds[[i]], xlim, ylim, cn = feat2, res = 140,
                main = sprintf("%s\ntest accuracy %.3f", names(preds)[i], accs[i]),
                xlab = "clump thickness", ylab = if (i == 1) "cell size" else "")
  points(jitter(te[[feat2[1]]]), jitter(te[[feat2[2]]]), col = pcol(te), pch = 19, cex = 0.6)
}



<p class="caption"><b>Figure 2.</b> Decision regions of the three families on the 205 test cases, each fitted with default settings to the 478 training cases. The accuracies in the titles are all close to 0.93 — the problem is easy enough that any sensible rule does well — but the boundaries differ in shape: naive Bayes draws a smooth curve (a conic section, Section 4.4), the linear support vector machine a straight line placed to maximise the gap between the classes (Section 7.1), and k-nearest neighbours an irregular frontier that follows the local majority (Section 6.4). The differences are invisible in the accuracy and decisive in how each method behaves away from the data.</p>

## 2. The probability needed for this unit

Two of the three families are built on probability, and the third is best understood through it (Section 6.2). This section reviews only the three facts that are needed, and introduces the running diagnostic example that will be solved by hand.

### 2.1 Joint, marginal and conditional probability

For two events \(A\) and \(B\) — say \(D\), "the patient has the disease", and \(T\), "the test is positive" — the **joint** probability \(P(D, T)\) is the probability that both occur; the **marginal** probability \(P(T)\) is the probability of \(T\) regardless of \(D\), obtained by summing the joint over the possibilities for \(D\) (the <em>law of total probability</em>, \(P(T) = P(T, D) + P(T, \neg D)\)); and the **conditional** probability \(P(T \mid D)\) is the probability of \(T\) among the cases where \(D\) holds, defined by

\[ P(T \mid D) = \frac{P(T, D)}{P(D)} \qquad\text{equivalently}\qquad P(T, D) = P(T \mid D)\, P(D). \]

The second form is the <em>product rule</em>, and it is symmetric: \(P(T, D) = P(D \mid T)\, P(T)\) as well. Two events are **independent** when \(P(T \mid D) = P(T)\), i.e. when knowing one tells nothing about the other, in which case the joint factorises as \(P(T, D) = P(T)\,P(D)\). Two events are **conditionally independent given** a third, \(C\), when \(P(A, B \mid C) = P(A \mid C)\, P(B \mid C)\): within each value of \(C\) they carry no information about each other, although they may be strongly associated overall. This last notion is the foundation of both naive Bayes and Bayesian networks.

### 2.2 Bayes' theorem, by hand

Equating the two forms of the product rule and dividing by \(P(T)\) gives the result that Thomas Bayes left in a manuscript published posthumously in 1763:<sup class="c"><a href="#r1">[1]</a></sup>

<div class="eq">\[ P(D \mid T) = \frac{P(T \mid D)\, P(D)}{P(T)} = \frac{P(T \mid D)\, P(D)}{P(T \mid D)\, P(D) + P(T \mid \neg D)\, P(\neg D)} \]
<span class="where">where \(P(D)\) is the <strong>prior</strong> probability of the disease before the test (in a screening setting, the prevalence), \(P(T \mid D)\) is the <strong>likelihood</strong> of the observed result under the disease (for a positive test, the sensitivity), \(P(T)\) is the <strong>evidence</strong> or marginal probability of the observed result, and \(P(D \mid T)\) is the <strong>posterior</strong> probability of the disease after the test — in clinical language, the positive predictive value (PPV).</span></div>

The theorem is an identity; its content lies in the direction of the inference. Test characteristics are measured in the forward direction (how does the test behave in people known to be ill?), whereas the clinician needs the backward direction (how likely is illness in a person whose test is positive?). The two directions are related only through the prior, and the neglect of the prior — <em>base-rate neglect</em> — is a well-documented error: when Casscells and colleagues asked staff and students at Harvard Medical School a version of the problem below, fewer than one in five gave the right answer, and the most common answer was off by a factor of five.<sup class="c"><a href="#r2">[2]</a></sup>

**Worked example.** A screening test for a disease with prevalence 1 % has sensitivity 95 % (\(P(T \mid D) = 0.95\)) and specificity 95 % (\(P(\neg T \mid \neg D) = 0.95\), so the false-positive rate is \(P(T \mid \neg D) = 0.05\)). A person tests positive. What is the probability that they have the disease?

\[ P(D \mid T) = \frac{0.95 \times 0.01}{0.95 \times 0.01 + 0.05 \times 0.99} = \frac{0.0095}{0.0095 + 0.0495} = \frac{0.0095}{0.0590} \approx 0.161. \]

A positive result from a test that is right 95 % of the time in both directions leaves the patient with only a 16 % probability of disease, because the 1 % of true cases who test positive (0.95 × 1 % ≈ 0.95 % of everyone) are outnumbered five to one by the 99 % of healthy people who test falsely positive (0.05 × 99 % ≈ 4.95 % of everyone). The same arithmetic is far easier to follow in <em>natural frequencies</em> — counts in a notional population of 10,000 — than in probabilities, as Gigerenzer and Hoffrage showed experimentally,<sup class="c"><a href="#r3">[3]</a></sup> and the chunk below lays it out both ways.



In [ ]:
# Bayes' theorem on the screening example, as probabilities and as natural frequencies in 10,000 people.
prevalence <- 0.01; sensitivity <- 0.95; specificity <- 0.95
p_pos_D <- sensitivity; p_pos_notD <- 1 - specificity
evidence <- p_pos_D * prevalence + p_pos_notD * (1 - prevalence)        # P(T) by total probability
ppv <- p_pos_D * prevalence / evidence
cat(sprintf("P(T) = %.4f   P(D | T) = PPV = %.4f\n", evidence, ppv))
N <- 10000; ill <- N * prevalence; healthy <- N * (1 - prevalence)
freq <- rbind("ill (100)"      = c(`test +` = ill * sensitivity,      `test -` = ill * (1 - sensitivity)),
              "healthy (9,900)" = c(`test +` = healthy * (1 - specificity), `test -` = healthy * specificity))
freq <- round(freq); print(freq)
cat(sprintf("Of the %d positives, %d are ill: %.3f\n", sum(freq[, 1]), freq[1, 1], freq[1, 1] / sum(freq[, 1])))
prev <- seq(0.001, 0.6, length.out = 400)
plot(prev * 100, 0.95 * prev / (0.95 * prev + 0.05 * (1 - prev)), type = "l", col = BLUE, lwd = 2,
     ylim = c(0, 1), xlab = "prevalence (%)", ylab = "P(disease | positive test)",
     main = "The posterior depends on the prior: PPV vs prevalence")
lines(prev * 100, 0.95 * prev / (0.95 * prev + 0.01 * (1 - prev)), col = GREEN, lwd = 2)
lines(prev * 100, 0.95 * prev / (0.95 * prev + 0.001 * (1 - prev)), col = ORANGE, lwd = 2)
abline(v = 1, col = GREY, lty = 3); points(1, ppv, pch = 19, col = BLUE)
legend("bottomright", c("specificity 0.95", "specificity 0.99", "specificity 0.999"),
       col = c(BLUE, GREEN, ORANGE), lwd = 2, bty = "n")
for (p in c(0.01, 0.10, 0.30))
  cat(sprintf("prevalence %2.0f%%: PPV = %.3f (specificity 0.95)\n", p * 100, 0.95 * p / (0.95 * p + 0.05 * (1 - p))))



The code reproduces the hand calculation: the PPV is 0.161. In the frequency table, 95 of the 100 ill people and 495 of the 9,900 healthy people test positive, so 95 of 590 positives are ill. The same test applied in a hospital ward where the prevalence is 30 % rather than 1 % would behave very differently; the figure traces the PPV over the whole range of prevalence and also shows what a more specific test buys.

<p class="caption"><b>Figure 3.</b> Positive predictive value against prevalence for a test with sensitivity 0.95 and three specificities. At 1 % prevalence the 95 %-specific test yields a PPV of 0.16; at 10 % it yields about 0.68 and at 30 % about 0.89 (printed values). Raising the specificity to 0.999 makes the test useful even in screening. The lesson transfers directly to classifiers: a model's sensitivity and specificity are properties of the model, but the meaning of its positive output depends on the population it is applied to, which is one reason the external-validation failures of Unit 1 happened in hospitals with a different case mix.</p>

<div class="box warn"><h4>Common misconception</h4><p>"The test is 95 % accurate, so a positive result means a 95 % chance of disease." The 95 % is \(P(T \mid D)\); the clinician needs \(P(D \mid T)\). The two are equal only when the prior is 50 %. Every probabilistic classifier in this unit outputs a posterior, and every posterior inherits the prior that was built into it — by the class proportions of the training sample. A naive Bayes model trained on a sample that is half malignant will overstate the probability of malignancy in a screening population; Section 3.1 shows where the prior enters.</p></div>

### 2.3 Bayes' theorem for classification

Replacing the binary event \(D\) with a class label \(y \in \{1, \dots, K\}\) and the test result with a feature vector \(\mathbf{x}\) gives the form used throughout this unit:

\[ P(y = c \mid \mathbf{x}) = \frac{p(\mathbf{x} \mid y = c)\, P(y = c)}{p(\mathbf{x})} = \frac{p(\mathbf{x} \mid y = c)\, P(y = c)}{\sum_{k=1}^{K} p(\mathbf{x} \mid y = k)\, P(y = k)}. \]

The lower-case \(p\) marks a probability <em>density</em>, because features are usually continuous: \(p(\mathbf{x} \mid y = c)\) is the <strong>class-conditional density</strong> — the distribution of the features among the cases of class \(c\) — and \(P(y = c)\) is the <strong>class prior</strong>. A generative classifier is a recipe for estimating these two objects from the training sample; once they are in hand, the posterior follows from the formula, and the denominator can be ignored when comparing classes because it is the same for all of them.

## 3. Bayesian decision theory

### 3.1 The Bayes classifier and the Bayes error

Suppose for a moment that the class-conditional densities and the priors are known exactly. Then the posterior \(P(y = c \mid \mathbf{x})\) is known for every \(\mathbf{x}\), and the rule that assigns each \(\mathbf{x}\) to the class of highest posterior,

\[ \hat{y}(\mathbf{x}) = \arg\max_c \, P(y = c \mid \mathbf{x}) = \arg\max_c \, p(\mathbf{x} \mid y = c)\, P(y = c), \]

is the **Bayes classifier**. It is optimal in a precise sense: no other rule has a lower probability of error. Its error, the **Bayes error**, is the irreducible error of the problem — the part caused by genuine overlap of the classes in feature space, which no amount of data or modelling can remove. Where the two classes have equal posterior, \(P(y = c \mid \mathbf{x}) = 1/2\) in the binary case, is the **Bayes decision boundary**; every classifier in this course can be read as an attempt to approximate it from a finite sample. The prior enters as a multiplicative weight on each class: doubling the prior of malignancy doubles its posterior odds everywhere, and moves the boundary toward the benign side.

### 3.2 Minimum-error and minimum-risk decisions

Choosing the most probable class minimises the <em>number</em> of errors. In medicine the two kinds of error rarely cost the same: a malignant tumour called benign is a missed cancer, a benign one called malignant is an unnecessary biopsy. Let \(\lambda_{\mathrm{FN}}\) be the cost of a false negative (deciding benign when the truth is malignant) and \(\lambda_{\mathrm{FP}}\) the cost of a false positive. The **expected risk** of each decision at a point \(\mathbf{x}\) is

\[ R(\text{decide benign} \mid \mathbf{x}) = \lambda_{\mathrm{FN}}\, P(M \mid \mathbf{x}), \qquad R(\text{decide malignant} \mid \mathbf{x}) = \lambda_{\mathrm{FP}}\, P(B \mid \mathbf{x}), \]

and the **minimum-risk** rule decides malignant when the first exceeds the second, i.e. when

\[ P(M \mid \mathbf{x}) > \frac{\lambda_{\mathrm{FP}}}{\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}}}. \]

With equal costs the threshold is \(1/2\) and the rule reduces to the minimum-error rule. If a missed cancer is judged five times as costly as an unnecessary biopsy, the threshold falls to \(1/6 \approx 0.167\): the classifier calls a case malignant as soon as the posterior exceeds one sixth. The chunk below builds the whole machinery on a single feature, clump thickness, by fitting one Gaussian per class and using the class proportions of the training sample as priors.



In [ ]:
# Bayesian decision theory in one dimension: Gaussian class-conditionals of clump thickness, priors, posterior, thresholds.
r_tr <- tr$Cl.thickness
prior <- c(malignant = mean(tr$Class == "malignant"), benign = mean(tr$Class == "benign"))
mu <- c(malignant = mean(r_tr[tr$Class == "malignant"]), benign = mean(r_tr[tr$Class == "benign"]))
sdv <- c(malignant = sd(r_tr[tr$Class == "malignant"]),  benign = sd(r_tr[tr$Class == "benign"]))
grid <- seq(0, 12, length.out = 1000)
jM <- dnorm(grid, mu["malignant"], sdv["malignant"]) * prior["malignant"]   # p(x|M) P(M)
jB <- dnorm(grid, mu["benign"],    sdv["benign"])    * prior["benign"]
postM <- jM / (jM + jB)
t_err  <- grid[which.min(abs(postM - 0.5))]
t_risk <- grid[which.min(abs(postM - 1/6))]
cat(sprintf("priors: malignant %.3f, benign %.3f | means: %.2f vs %.2f, sd: %.2f vs %.2f\n",
            prior["malignant"], prior["benign"], mu["malignant"], mu["benign"], sdv["malignant"], sdv["benign"]))
cat(sprintf("clump-thickness threshold: minimum-error %.2f | minimum-risk (missed cancer 5x costlier) %.2f\n", t_err, t_risk))
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
plot(grid, jM, type = "l", col = ORANGE, lwd = 2, xlab = "clump thickness", ylab = "density x prior",
     main = "Class-conditional densities x priors", ylim = c(0, max(jM, jB)))
lines(grid, jB, col = BLUE, lwd = 2)
polygon(c(grid, rev(grid)), c(pmin(jM, jB), rep(0, length(grid))), col = adjustcolor(GREY, 0.35), border = NA)
legend("topright", c("p(x|M) P(M)", "p(x|B) P(B)", "Bayes error (overlap)"),
       col = c(ORANGE, BLUE, GREY), lwd = c(2, 2, 6), bty = "n", cex = 0.8)
plot(grid, postM, type = "l", col = ORANGE, lwd = 2, xlab = "clump thickness", ylab = "P(malignant | x)",
     main = "Posterior and the two decision thresholds")
abline(h = 0.5, lty = 3); abline(v = t_err, lty = 2)
abline(h = 1/6, col = GREEN, lty = 3); abline(v = t_risk, col = GREEN, lty = 2)
legend("bottomright", c(sprintf("equal costs: %.1f", t_err), sprintf("FN costs 5x FP: %.1f", t_risk)),
       col = c("black", GREEN), lty = 2, bty = "n", cex = 0.8)
# Consequences on the 205 test cases: errors of each kind and total cost (FN = 5, FP = 1).
rows <- lapply(list(c("minimum error (threshold 1/2)", t_err), c("minimum risk (threshold 1/6)", t_risk)),
  function(z) {
    t <- as.numeric(z[2]); callM <- te$Cl.thickness >= t
    FN <- sum(!callM & te$Class == "malignant"); FP <- sum(callM & te$Class == "benign")
    data.frame(rule = z[1], FN = FN, FP = FP, errors = FN + FP,
               sens = round(mean(callM[te$Class == "malignant"]), 3),
               spec = round(mean(!callM[te$Class == "benign"]), 3), cost_5FN_FP = 5 * FN + FP)
  })
print(do.call(rbind, rows), row.names = FALSE)



<p class="caption"><b>Figure 4.</b> Left: the two class-conditional Gaussians of clump thickness, each multiplied by its prior, so that the curves are proportional to the posterior; the shaded overlap is the region in which the Bayes classifier must err. Right: the posterior probability of malignancy. The equal-cost threshold is where the posterior crosses 0.5; when a missed cancer is five times as costly as an unnecessary biopsy, the threshold moves to where the posterior crosses 1/6 (printed values), and more cases are called malignant. The malignant Gaussian is wider than the benign one (printed standard deviations), so far from both means its tails exceed the benign density — the same variance effect that decides the hand example of Section 4.3; in practice one would not call a very thin, very regular nucleus malignant, a reminder that a model is only trustworthy where there are data.</p>

The two thresholds trade one kind of error for the other. The table printed above the figure applies both to the 205 test cases and counts the consequences, including the total cost under the 5 : 1 assumption. The minimum-error rule makes fewer errors in total; the minimum-risk rule makes more errors but fewer of the expensive kind, and its total cost under the stated costs is lower. Neither rule is "better" in the abstract: the choice of costs is a clinical and ethical decision that the engineer must elicit and make explicit, not a modelling detail. Supplement S1 generalises this trade-off into the ROC curve, which displays all thresholds at once.

<div class="box key"><h4>Key idea</h4><p>Bayesian decision theory separates two things that are often confused: <em>estimating</em> the posterior probability of each class, which is a statistical problem, and <em>deciding</em> what to do with it, which depends on the costs of the possible errors. A classifier that outputs calibrated probabilities can serve any cost structure by moving the threshold; one that outputs only labels has baked in the assumption of equal costs.</p></div>

## 4. Naive Bayes

### 4.1 The conditional-independence assumption

To use Bayes' theorem for classification one must estimate the class-conditional density \(p(\mathbf{x} \mid y = c)\). For a single feature this is easy: a histogram or a fitted Gaussian will do. For many features it is hopeless in general, because a joint density over nine (or thirty) dimensions has far too many degrees of freedom to be estimated from a few hundred cases — the curse of dimensionality that Section 6.5 will quantify. The **naive Bayes** classifier escapes by assuming that, <em>within each class</em>, the features are independent of one another:

<div class="eq">\[ p(\mathbf{x} \mid y = c) = \prod_{j=1}^{p} p(x_j \mid y = c), \qquad\text{so that}\qquad P(y = c \mid \mathbf{x}) \propto P(y = c) \prod_{j=1}^{p} p(x_j \mid y = c). \]
<span class="where">The p-dimensional density is replaced by p one-dimensional densities per class, each of which can be estimated from the data of that class alone. The classifier then needs only the per-class prior and, for each feature, a one-dimensional model of its distribution in each class.</span></div>

The assumption is almost always false — in the breast-cancer data, cell size and cell shape of the same nucleus are strongly correlated within each class — which is why the method is called <em>naive</em>, or in Hand and Yu's phrase, "idiot's Bayes".<sup class="c"><a href="#r6">[6]</a></sup> Section 4.6 explains why it works anyway. In practice the product is computed as a sum of logarithms, because the product of many densities underflows the floating-point range:

\[ \log P(y = c \mid \mathbf{x}) = \log P(y = c) + \sum_{j=1}^{p} \log p(x_j \mid y = c) + \text{const}. \]

The choice of the one-dimensional model \(p(x_j \mid y = c)\) gives the variants of the method. **Gaussian naive Bayes**, for continuous features, fits a normal distribution \(\mathcal{N}(\mu_{cj}, \sigma^2_{cj})\) to each feature in each class, so training consists of computing a mean and a variance per feature and class, plus the class proportions. **Multinomial naive Bayes**, for count features such as word frequencies in a document, models each class by a probability vector over the vocabulary; **Bernoulli naive Bayes**, for binary features (word present or absent), models each feature by a per-class probability of being 1. The last two are the classical spam filter and the baseline of text classification, including the classification of clinical notes; Section 4.7 shows them on image data. In R the Gaussian and categorical variants are provided by <code>e1071::naiveBayes</code>;<sup class="c"><a href="#r37">[37]</a></sup> the count variants are written from scratch below.

Naive Bayes has three practical virtues: it is <strong>easy to build</strong> (no iterative optimisation, no hyper-parameters beyond a smoothing constant), it is <strong>efficient</strong> (training is a single pass over the data, prediction is a sum of \(p\) terms), and it performs well on <strong>very large and high-dimensional data</strong>, including noisy data, because the number of parameters grows only linearly with the number of features.

### 4.2 Naive Bayes by hand: the weather example

The classical hand example, used in the lecture slides and in most textbooks, is Quinlan's fourteen-day "play tennis" dataset: four categorical features — Outlook (Sunny / Overcast / Rainy), Temperature (Hot / Mild / Cool), Humidity (High / Normal) and Windy (True / False) — and a binary target, whether the match was played (Yes on 9 days, No on 5).<sup class="c"><a href="#r12">[12]</a></sup> The computation has three steps.

**Step 1 — frequency table.** Count, for each value of Outlook, how many days fell in each class:

| Outlook | No | Yes | total |
|---|---|---|---|
| Overcast | 0 | 4 | 4 |
| Rainy | 3 | 2 | 5 |
| Sunny | 2 | 3 | 5 |
| **total** | **5** | **9** | **14** |

**Step 2 — likelihood table.** Convert the counts to probabilities: the prior \(P(\text{Yes}) = 9/14 = 0.64\), the evidence \(P(\text{Sunny}) = 5/14 = 0.36\), and the likelihood \(P(\text{Sunny} \mid \text{Yes}) = 3/9 = 0.33\).

**Step 3 — posterior.** Bayes' theorem gives \(P(\text{Yes} \mid \text{Sunny}) = \dfrac{P(\text{Sunny} \mid \text{Yes})\, P(\text{Yes})}{P(\text{Sunny})} = \dfrac{0.33 \times 0.64}{0.36} = 0.60\), so on a sunny day the prediction is **play**. (Exactly, \(\tfrac{3}{9} \cdot \tfrac{9}{14} \big/ \tfrac{5}{14} = 3/5\).)

With all four features the "naive" step enters: the likelihood of the whole day is the product of the per-feature likelihoods. **Query:** Outlook = Rainy, Temperature = Mild, Humidity = Normal, Windy = True. The unnormalised posteriors are

\( P(\text{Yes}) \prod_j P(x_j \mid \text{Yes}) = \tfrac{9}{14} \times \tfrac{2}{9} \times \tfrac{4}{9} \times \tfrac{6}{9} \times \tfrac{3}{9} \approx 0.01411 \)

\( P(\text{No}) \prod_j P(x_j \mid \text{No}) = \tfrac{5}{14} \times \tfrac{3}{5} \times \tfrac{2}{5} \times \tfrac{1}{5} \times \tfrac{3}{5} \approx 0.01029 \)

and the **normalisation** divides each by their sum, \(0.01411 + 0.01029 = 0.02440\) (which is \(p(\mathbf{x})\), the evidence): \(P(\text{Yes} \mid \mathbf{x}) \approx 0.578\) and \(P(\text{No} \mid \mathbf{x}) \approx 0.422\). The prediction is **play**, with moderate confidence. Note the zero in the Overcast row of the No column: any overcast day would receive \(P(\text{No} \mid \mathbf{x}) = 0\) regardless of the other three features, because a single zero annihilates the product. This is the <em>zero-frequency problem</em>, and the standard remedy, **Laplace (additive) smoothing**, adds a pseudo-count \(\alpha\) to every cell before dividing. The chunk builds the fourteen days, repeats the computation in base R, and then with <code>e1071::naiveBayes</code>, first without smoothing (reproducing the hand result) and then with \(\alpha = 1\).

<div class="box warn"><h4>Where the slides differ from the original data</h4><p>The frequency table above is the one in the lecture slides and in many online tutorials. In Quinlan's original table the Sunny and Rainy counts are the other way round (Sunny: 2 Yes, 3 No; Rainy: 3 Yes, 2 No),<sup class="c"><a href="#r12">[12]</a></sup> so that the original answer for a sunny day is \(P(\text{Yes} \mid \text{Sunny}) = 2/5 = 0.40\) — do not play. The arithmetic of the method is identical; only the labels of two outlook values are exchanged. The notebook follows the slides' table so that the numbers match the lecture; the dataset built below is Quinlan's with those two labels swapped.</p></div>



In [ ]:
# The fourteen-day weather dataset (as tabulated in the lecture slides), naive Bayes by hand, then with e1071.
weather <- data.frame(
  Outlook     = c("Rainy","Rainy","Overcast","Sunny","Sunny","Sunny","Overcast","Rainy","Rainy","Sunny","Rainy","Overcast","Overcast","Sunny"),
  Temperature = c("Hot","Hot","Hot","Mild","Cool","Cool","Cool","Mild","Cool","Mild","Mild","Mild","Hot","Mild"),
  Humidity    = c("High","High","High","High","Normal","Normal","Normal","High","Normal","Normal","Normal","High","Normal","High"),
  Windy       = factor(c(FALSE,TRUE,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,TRUE)),
  Play        = c("No","No","Yes","Yes","Yes","No","Yes","No","Yes","Yes","Yes","Yes","Yes","No"),
  stringsAsFactors = TRUE)
cat("Step 1 - frequency table for Outlook:\n"); print(addmargins(table(Outlook = weather$Outlook, Play = weather$Play)))
prior <- prop.table(table(weather$Play))                               # P(Yes), P(No)
lik <- function(feature, value, cls) mean(weather[[feature]][weather$Play == cls] == value)
cat(sprintf("\nStep 2 - P(Sunny) = %.2f, P(Yes) = %.2f, P(Sunny | Yes) = %.2f\n",
            mean(weather$Outlook == "Sunny"), prior["Yes"], lik("Outlook", "Sunny", "Yes")))
cat(sprintf("Step 3 - P(Yes | Sunny) = %.2f\n\n", lik("Outlook", "Sunny", "Yes") * prior["Yes"] / mean(weather$Outlook == "Sunny")))
q <- list(Outlook = "Rainy", Temperature = "Mild", Humidity = "Normal", Windy = "TRUE")
unnorm <- sapply(c("Yes", "No"), function(c) prior[[c]] * prod(mapply(function(f, v) lik(f, v, c), names(q), q)))
cat("by hand, query (Rainy, Mild, Normal, Windy):\n")
for (c in c("Yes", "No")) cat(sprintf("  P(%s) x likelihoods = %.5f  ->  normalised P(%s | x) = %.3f\n",
                                       c, unnorm[c], c, unnorm[c] / sum(unnorm)))
qdf <- data.frame(Outlook = factor("Rainy", levels(weather$Outlook)), Temperature = factor("Mild", levels(weather$Temperature)),
                  Humidity = factor("Normal", levels(weather$Humidity)), Windy = factor("TRUE", levels(weather$Windy)))
cat(sprintf("\ne1071 naiveBayes: alpha=0 (laplace=0) P(Yes|x) = %.3f  |  alpha=1 (laplace=1) P(Yes|x) = %.3f\n",
            predict(naiveBayes(Play ~ ., weather, laplace = 0), qdf, type = "raw", threshold = 0)[1, "Yes"],
            predict(naiveBayes(Play ~ ., weather, laplace = 1), qdf, type = "raw")[1, "Yes"]))



With \(\alpha = 0\), <code>e1071::naiveBayes</code> returns the hand result, 0.578. With \(\alpha = 1\) the posterior changes slightly to 0.586, because every likelihood is pulled toward uniform and, more importantly, the impossible \(P(\text{Overcast} \mid \text{No}) = 0\) becomes positive, so that an overcast day is no longer assigned to Yes with certainty. Smoothing costs a little fidelity to the sample and buys robustness to the values the sample happened not to contain; with 14 days the effect is visible, with 14,000 it would be negligible.

### 4.3 Gaussian naive Bayes by hand

The weather example used categorical features, for which a likelihood is a relative frequency. For continuous features a density model is needed, and the Gaussian variant supplies it. Take the six training cases of Unit 1, with two features \(x_1\) and \(x_2\), and the same new case \(\mathbf{x} = (15, 4)\) that the nearest-centroid classifier called benign:

| case | \(x_1\) | \(x_2\) | class |
|---|---|---|---|
| 1 | 12 | 2 | benign |
| 2 | 13 | 3 | benign |
| 3 | 11 | 1 | benign |
| 4 | 19 | 9 | malignant |
| 5 | 21 | 11 | malignant |
| 6 | 17 | 7 | malignant |

**Training.** The priors are \(P(B) = P(M) = 3/6 = 0.5\). The per-class means are \(\boldsymbol{\mu}_B = (12, 2)\) and \(\boldsymbol{\mu}_M = (19, 9)\). Using the <em>population</em> variance (dividing by \(n_c\), as scikit-learn does), the benign variances are \(\sigma^2_{B1} = \tfrac{0 + 1 + 1}{3} = \tfrac{2}{3}\) and \(\sigma^2_{B2} = \tfrac{2}{3}\); the malignant variances are \(\sigma^2_{M1} = \tfrac{0 + 4 + 4}{3} = \tfrac{8}{3}\) and \(\sigma^2_{M2} = \tfrac{8}{3}\). The malignant class is four times more dispersed than the benign one in both features.

**Classification** of \(\mathbf{x} = (15, 4)\) uses the log-density \(\log \mathcal{N}(x \mid \mu, \sigma^2) = -\tfrac{1}{2}\log(2\pi\sigma^2) - \tfrac{(x-\mu)^2}{2\sigma^2}\). Adding the log-prior \(\log 0.5\) to each sum gives \(\log P(B \mid \mathbf{x}) + \text{const} \approx -11.88\) and \(\log P(M \mid \mathbf{x}) + \text{const} \approx -11.20\). The malignant class has the larger log-posterior, so naive Bayes classifies the case **malignant** — the opposite of the nearest-centroid answer of Unit 1, although the case is closer to the benign centroid. Normalising, \(P(M \mid \mathbf{x}) = \tfrac{1}{1 + e^{-0.676}} \approx 0.663\). The reason is the variance: a point 3 units from the tight benign cluster is more surprising, under a Gaussian model, than a point 4–5 units from the diffuse malignant cluster. The chunk confirms the arithmetic step by step.



In [ ]:
# Gaussian naive Bayes by hand on the six cases (population variance), then e1071 for comparison.
six <- rbind(c(12,2), c(13,3), c(11,1), c(19,9), c(21,11), c(17,7)); colnames(six) <- c("x1", "x2")
cls <- factor(c("benign","benign","benign","malignant","malignant","malignant"))
newx <- c(x1 = 15, x2 = 4)
log_gauss <- function(x, m, v) -0.5 * log(2 * pi * v) - (x - m)^2 / (2 * v)
logpost <- sapply(levels(cls), function(c) {
  Xc <- six[cls == c, , drop = FALSE]; m <- colMeans(Xc); v <- colMeans(sweep(Xc, 2, m)^2)   # population variance
  terms <- log_gauss(newx, m, v)
  cat(sprintf("%-9s mean (%.0f,%.0f) var (%.2f,%.2f)  log p(x_j|c)=(%.3f,%.3f)  log-prior+sum=%.3f\n",
              c, m[1], m[2], v[1], v[2], terms[1], terms[2], log(mean(cls == c)) + sum(terms)))
  log(mean(cls == c)) + sum(terms)
})
pM <- 1 / (1 + exp(logpost["benign"] - logpost["malignant"]))
cat(sprintf("by hand:  P(malignant | x) = %.4f,  P(benign | x) = %.4f\n", pM, 1 - pM))
nb6 <- naiveBayes(Class ~ ., data.frame(six, Class = cls))
cat(sprintf("e1071 naiveBayes (sample variance, n-1): P(malignant | x) = %.4f\n",
            predict(nb6, data.frame(t(newx)), type = "raw")[1, "malignant"]))



The hand calculation gives \(P(M \mid \mathbf{x}) = 0.6629\), exactly the Python result. <code>e1071::naiveBayes</code> returns 0.497 — a near-tie that leans benign — and the discrepancy is instructive and worth keeping in mind.

<div class="box warn"><h4>Caveat: n versus n − 1</h4><p>scikit-learn's <code>GaussianNB</code> uses the <em>population</em> variance (divides the sum of squared deviations by \(n_c\)); <code>e1071::naiveBayes</code> uses the <em>sample</em> variance (divides by \(n_c - 1\)). With only three cases per class the difference is large — it widens every Gaussian by a factor \(3/2\) — and it is enough to move this borderline point from malignant (0.66, population) to a near-tie leaning benign (0.50, sample). With the hundreds of cases per class of the real data (Section 4.4) the factor \(n_c/(n_c-1)\) is negligible and the two conventions agree to three decimals. The lesson is not that one convention is wrong but that a posterior sitting on the boundary is sensitive to small modelling choices — exactly the overlap region where no method can be confident.</p></div>

### 4.4 A naive Bayes classifier from scratch, compared with e1071

The six-case computation generalises without change to \(n\) cases and \(p\) features. The function below is a complete Gaussian naive Bayes classifier in about fifteen lines: <code>fit</code> stores the log-priors, means and variances; <code>predict</code> sums the per-feature log-densities and normalises with the log-sum-exp trick. It is tested on all nine features of the breast-cancer data against <code>e1071::naiveBayes</code>.



In [ ]:
# A Gaussian naive Bayes classifier in base R, and a comparison with e1071 on all nine features.
hand_gnb_fit <- function(X, y, pop = TRUE) {
  cl <- levels(y); denom <- if (pop) 1 else NULL
  list(cl = cl, logprior = log(sapply(cl, function(c) mean(y == c))),
       mean = t(sapply(cl, function(c) colMeans(X[y == c, , drop = FALSE]))),
       var  = t(sapply(cl, function(c) { Xc <- X[y == c, , drop = FALSE]
                 colSums(sweep(Xc, 2, colMeans(Xc))^2) / (nrow(Xc) - !pop) })))
}
hand_gnb_logpost <- function(m, X) {
  ll <- sapply(seq_along(m$cl), function(k)
    rowSums(-0.5 * log(2 * pi * matrix(m$var[k, ], nrow(X), ncol(X), byrow = TRUE)) -
            sweep(X, 2, m$mean[k, ])^2 / (2 * matrix(m$var[k, ], nrow(X), ncol(X), byrow = TRUE))))
  joint <- sweep(ll, 2, m$logprior, "+")
  out <- joint - apply(joint, 1, function(r) log(sum(exp(r - max(r)))) + max(r))   # normalise over classes
  colnames(out) <- m$cl; out
}
Xall <- as.matrix(bc[, 1:9]); ytr <- bc$Class[idx]
Xtr <- as.matrix(tr[, 1:9]); Xte <- as.matrix(te[, 1:9])
hand <- hand_gnb_fit(Xtr, tr$Class, pop = TRUE)
P_hand <- exp(hand_gnb_logpost(hand, Xte))
nb9 <- naiveBayes(tr[, 1:9], tr$Class)
P_e1071 <- predict(nb9, Xte, type = "raw")[, levels(bc$Class)]
cat(sprintf("largest difference in any posterior, hand vs e1071: %.2e  (negligible: both use hundreds of cases per class)\n",
            max(abs(P_hand[, hand$cl] - P_e1071[, hand$cl]))))
pred_hand <- factor(hand$cl[max.col(hand_gnb_logpost(hand, Xte))], levels = levels(bc$Class))
cat(sprintf("test accuracy, 9 features: hand-written %.3f | e1071 %.3f\n",
            mean(pred_hand == te$Class), mean(predict(nb9, Xte) == te$Class)))
cat(sprintf("test accuracy, 2 features: e1071 %.3f\n", mean(predict(nb_p, te) == te$Class)))



The two implementations agree to within 0.007 on every posterior — the small residue of the population-versus-sample variance convention (Section 4.3), negligible with hundreds of cases per class — and reach the same accuracy on the held-out cases: above 0.95 with the nine features, and slightly lower with the two features of the pictures. Training took one pass over the data to compute nine means and nine variances per class — naive Bayes is the fastest classifier to train in this course, which is one reason it remains the baseline for very large problems.

The shape of the boundary follows from the model. For two classes with Gaussian features, the log-posterior ratio is a quadratic function of \(\mathbf{x}\) (the squared terms \((x_j - \mu_{cj})^2 / 2\sigma^2_{cj}\) have different coefficients in the two classes when the variances differ), so the boundary where it vanishes is a conic section: an ellipse, parabola or hyperbola. If the variances were equal in both classes the quadratic terms would cancel and the boundary would be a straight line — the same linear form as logistic regression, reached by a different route. The chunk below draws the fitted class-conditional Gaussians and the boundary they produce on the two-feature problem.



In [ ]:
# The Gaussian model naive Bayes fits in two dimensions, and the (quadratic) boundary it implies.
shade_regions(function(G) predict(nb_p, G) == "malignant", xlim, ylim, cn = feat2, res = 150,
              main = "Gaussian naive Bayes: per-class Gaussians (1,2 sd) and boundary",
              xlab = "clump thickness", ylab = "cell size")
points(jitter(te[[feat2[1]]]), jitter(te[[feat2[2]]]), col = pcol(te), pch = 19, cex = 0.55)
th <- seq(0, 2 * pi, length.out = 100)
for (c in levels(tr$Class)) {
  m <- colMeans(tr[tr$Class == c, feat2]); s <- apply(tr[tr$Class == c, feat2], 2, sd)
  col <- if (c == "malignant") ORANGE else BLUE
  for (nsd in 1:2) lines(m[1] + nsd * s[1] * cos(th), m[2] + nsd * s[2] * sin(th), col = col, lwd = 1.3, lty = nsd)
  points(m[1], m[2], pch = 4, col = col, lwd = 3, cex = 1.6)
}
legend("topleft", c("malignant", "benign"), col = c(ORANGE, BLUE), pch = 19, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 5.</b> The model behind the boundary: one axis-aligned Gaussian per class (solid, 1 standard deviation; dashed, 2), with centres marked by crosses, and the resulting decision regions with the test cases on top. The malignant class has much larger variance in both features, so its Gaussian is wider; the boundary bends around the tight benign cluster and is a curve, not a line. The ellipses are axis-aligned because naive Bayes assumes no correlation between the two features within a class — visibly false for the malignant cases, which trend along the diagonal, yet the boundary is still sensible.</p>

### 4.5 Generative versus discriminative

Naive Bayes and logistic regression (Unit 2) can produce the same family of boundaries, and the comparison between them is instructive. Logistic regression is <em>discriminative</em>: it models \(P(y \mid \mathbf{x})\) directly and fits its coefficients to make the training labels as probable as possible, with no model of how \(\mathbf{x}\) is distributed. Naive Bayes is <em>generative</em>: it models \(p(\mathbf{x} \mid y)\) and \(P(y)\) and obtains \(P(y \mid \mathbf{x})\) by Bayes' theorem. Ng and Jordan showed, theoretically and on many datasets, that the two have different learning curves: naive Bayes approaches its asymptotic error with far fewer training cases, but that asymptote is higher, because its assumptions bias it; logistic regression needs more data and ends up better.<sup class="c"><a href="#r7">[7]</a></sup> With a few dozen cases, prefer naive Bayes; with thousands, prefer the discriminative model. In medical problems, where labelled cases are expensive, the small-sample regime is common, and naive Bayes has a long record in computer-aided diagnosis for exactly that reason.<sup class="c"><a href="#r8">[8]</a></sup>

### 4.6 Why naive Bayes works although its assumption is false

The puzzle is empirical: naive Bayes classifies well on data in which the features are strongly dependent. Domingos and Pazzani resolved it in 1997.<sup class="c"><a href="#r5">[5]</a></sup> Classification under zero-one loss depends only on which class has the largest posterior, not on the value of the posterior; the independence assumption can distort the posterior probabilities considerably without changing their ranking. They proved that naive Bayes is optimal for some problems in which the assumption is grossly violated, and showed on 28 datasets that its accuracy was not related to the degree of dependence among the features. Hand and Yu added the statistical argument that the simple model has few parameters and therefore low variance: its bias is real, but with small samples the variance of a richer model is worse.<sup class="c"><a href="#r6">[6]</a></sup> The price of the assumption is paid elsewhere: the posteriors are badly <em>calibrated</em>, typically pushed toward 0 and 1 because every correlated feature counts as independent evidence. The chunk below exhibits both halves of the story.



In [ ]:
# Half 1: strongly dependent features do not ruin the ranking (5-fold CV accuracy with redundant features).
cv_acc <- function(fitpred, X, y, K = 5) {
  set.seed(1); folds <- sample(rep(1:K, length.out = nrow(X)))
  mean(sapply(1:K, function(k) {
    tr_i <- folds != k; p <- fitpred(X[tr_i, , drop = FALSE], y[tr_i], X[!tr_i, , drop = FALSE])
    mean(p == y[!tr_i]) }))
}
nb_fp <- function(Xtr, ytr, Xte) predict(naiveBayes(Xtr, ytr), Xte)
size3 <- c("Cell.size", "Cell.shape", "Epith.c.size")                 # strongly correlated size features
cat("Correlation among three size features (malignant class):\n")
print(round(cor(bc[bc$Class == "malignant", size3]), 3))
for (lbl in list(c("3 correlated size features", list(size3)), c("all 9 features", list(names(bc)[1:9]))))
  cat(sprintf("NB 5-fold CV accuracy with %-28s: %.3f\n", lbl[[1]], cv_acc(nb_fp, bc[, lbl[[2]], drop = FALSE], bc$Class)))
# Half 2: the posteriors are over-confident. Fraction of test cases with an extreme posterior, NB vs logistic regression.
glm_fit <- suppressWarnings(glm(Class ~ ., data = tr, family = binomial))
p_nb  <- apply(predict(nb9, Xte, type = "raw"), 1, max)
p_lr  <- predict(glm_fit, te, type = "response"); p_lr <- pmax(p_lr, 1 - p_lr)
cat(sprintf("\nGaussian naive Bayes: accuracy %.3f | %.0f%% of test cases get posterior > 0.99 | mean max-posterior %.3f\n",
            mean(predict(nb9, Xte) == te$Class), 100 * mean(p_nb > 0.99), mean(p_nb)))
cat(sprintf("logistic regression : accuracy %.3f | %.0f%% of test cases get posterior > 0.99 | mean max-posterior %.3f\n",
            mean(ifelse(predict(glm_fit, te, type = "response") > 0.5, "malignant", "benign") == te$Class),
            100 * mean(p_lr > 0.99), mean(p_lr)))



Cell size and cell shape are correlated above 0.7 within a class, and all three size features are mutually associated (printed table); treating them as independent violates the assumption, yet using all nine features — several of them redundant — does not harm the cross-validated accuracy (0.96 against 0.95 for the three features alone). The second half shows the price: naive Bayes assigns a posterior above 0.99 to a large fraction of the test cases, more often than logistic regression does, because each redundant feature is counted as fresh evidence. When the output of a classifier is to be used as a probability — to set a cost-sensitive threshold as in Section 3.2, or to be shown to a clinician — naive Bayes posteriors should be re-calibrated on held-out data (Supplement S1 discusses calibration), or a discriminative model should be preferred.

<div class="box warn"><h4>Common misconception</h4><p>"Naive Bayes is accurate, so its probabilities can be trusted." Accuracy depends on the ranking of the posteriors; calibration depends on their values. Naive Bayes often has the first without the second. A 0.99 from a naive Bayes model on correlated features is not a 99 % chance of malignancy; it is a vote in which the same witness was counted several times.</p></div>

### 4.7 Naive Bayes for counts and binary features

The Gaussian variant suits continuous measurements. For count data — words in a clinical note, reads mapped to a gene, pixels in a grey-level image — the multinomial variant models each class by a vector of category probabilities estimated as smoothed frequencies, and the Bernoulli variant models the presence or absence of each feature. Because R has no built-in digits set, the chunk below <em>simulates</em> 8 × 8 images of three classes (a vertical bar, a horizontal bar and a diagonal, each with noise), treats each image as 64 counts (multinomial) or 64 binary "ink present" features (Bernoulli), and writes both variants from scratch; it also fits the Gaussian variant for comparison. The point is not the accuracy but that the same algorithm serves three data types by changing only the one-dimensional model.



In [ ]:
# Simulated 8x8 digit-like images; from-scratch multinomial and Bernoulli naive Bayes, plus Gaussian.
set.seed(0); npc <- 120; patterns <- list(vbar = NULL, hbar = NULL, diag = NULL)
base_imgs <- function(kind) {
  m <- matrix(1, 8, 8)                                                   # faint background ink
  if (kind == "vbar") m[, 4:5] <- 8 else if (kind == "hbar") m[4:5, ] <- 8 else diag(m) <- 8
  as.vector(m)
}
Xdig <- do.call(rbind, lapply(names(patterns), function(k)
  t(replicate(npc, pmax(0, round(base_imgs(k) + rpois(64, 1) - 1))))))
ydig <- factor(rep(names(patterns), each = npc))
par(mfrow = c(1, 6), mar = c(0.2, 0.2, 1.4, 0.2))
for (i in c(1, 2, npc + 1, npc + 2, 2 * npc + 1, 2 * npc + 2))
  image(matrix(Xdig[i, ], 8)[, 8:1], col = grey.colors(12, 1, 0), axes = FALSE, main = as.character(ydig[i]))
# from-scratch count/binary naive Bayes with Laplace smoothing
multinom_nb <- function(Xtr, ytr, Xte, binary = FALSE) {
  if (binary) { Xtr <- (Xtr > 3) + 0; Xte <- (Xte > 3) + 0 }
  cl <- levels(ytr); lp <- log(table(ytr) / length(ytr))
  if (binary) {
    theta <- sapply(cl, function(c) (colSums(Xtr[ytr == c, ]) + 1) / (sum(ytr == c) + 2))
    score <- sapply(cl, function(c) Xte %*% log(theta[, c]) + (1 - Xte) %*% log(1 - theta[, c]) + lp[c])
  } else {
    theta <- sapply(cl, function(c) (colSums(Xtr[ytr == c, ]) + 1) / (sum(Xtr[ytr == c, ]) + ncol(Xtr)))
    score <- sapply(cl, function(c) Xte %*% log(theta[, c]) + lp[c])
  }
  factor(cl[max.col(score)], levels = cl)
}
set.seed(2); ii <- sample(nrow(Xdig)); cut <- floor(0.7 * nrow(Xdig))
trI <- ii[1:cut]; teI <- ii[(cut + 1):nrow(Xdig)]
cat(sprintf("multinomial NB (counts)  test accuracy: %.3f\n", mean(multinom_nb(Xdig[trI, ], ydig[trI], Xdig[teI, ]) == ydig[teI])))
cat(sprintf("Bernoulli   NB (ink>3)   test accuracy: %.3f\n", mean(multinom_nb(Xdig[trI, ], ydig[trI], Xdig[teI, ], binary = TRUE) == ydig[teI])))
cat(sprintf("Gaussian    NB (e1071)   test accuracy: %.3f\n", mean(predict(naiveBayes(Xdig[trI, ], ydig[trI]), Xdig[teI, ]) == ydig[teI])))



<p class="caption"><b>Figure 5b.</b> Two simulated examples of each of the three image classes (vertical bar, horizontal bar, diagonal), with Poisson noise added to every pixel.</p>

The multinomial and Bernoulli variants, written in a few lines, classify the three simulated classes essentially perfectly with a single pass over the data; the Gaussian variant does comparably here because the classes are well separated. Choosing the variant whose one-dimensional model matches the feature type is the only modelling decision naive Bayes asks of the analyst.

<div class="exercise"><h4>Exercise 1</h4><p>In the chunk of Section 4.4, train the hand-written and <code>e1071</code> classifiers on only the first 20 training cases (<code>tr[1:20, ]</code>) and compare their test accuracy with the logistic-regression model of Section 4.6 trained on the same 20 cases. Then repeat with 200 cases. Which model wins in each regime, and how does this relate to the result of Ng and Jordan?</p></div>

## 5. Bayesian networks

### 5.1 From the joint distribution to a graph

Naive Bayes makes one extreme assumption — all features independent given the class — to make a joint distribution tractable. A **Bayesian network** (belief network, probabilistic graphical model) makes the assumptions explicit and selective: it is a directed acyclic graph (DAG) whose nodes are random variables and whose arrows encode direct dependence, together with, for each node, a **conditional probability table (CPT)** giving the distribution of the node for each combination of values of its parents. The semantics of the graph is a single equation, the **factorisation of the joint distribution**:

<div class="eq">\[ P(X_1, X_2, \dots, X_m) = \prod_{i=1}^{m} P\big(X_i \mid \mathrm{parents}(X_i)\big) \]
<span class="where">Each variable depends directly only on its parents in the graph. The equation follows from the chain rule of probability, \(P(X_1, \dots, X_m) = \prod_i P(X_i \mid X_1, \dots, X_{i-1})\), once each variable is assumed independent of its non-descendants given its parents (the <em>local Markov property</em>).</span></div>

The formalism was developed by Judea Pearl in the 1980s as a representation of uncertain knowledge for expert systems, replacing the ad-hoc certainty factors of MYCIN (Unit 1) with coherent probability,<sup class="c"><a href="#r9">[9]</a>,<a href="#r10">[10]</a></sup> and the first efficient general inference algorithm was published by Lauritzen and Spiegelhalter in 1988 with a medical example that has become the standard teaching network: a fictitious chest clinic in which a visit to Asia, smoking, tuberculosis, lung cancer and bronchitis explain a chest X-ray and dyspnoea.<sup class="c"><a href="#r11">[11]</a></sup> This section uses a reduced version of that network with five binary variables — <em>Smoking</em> (S), lung <em>Cancer</em> (C), <em>Bronchitis</em> (B), positive chest <em>X-ray</em> (X) and <em>Dyspnoea</em> (D) — keeping the published conditional probabilities for the retained arrows. The chunk below draws it and prints the CPTs.



In [ ]:
# The reduced 'Asia' network: a DAG with five binary variables, drawn with base graphics; CPTs printed.
pos <- list(S = c(0.5, 0.9), C = c(0.25, 0.55), B = c(0.75, 0.55), X = c(0.1, 0.15), D = c(0.6, 0.15))
lab <- list(S = "Smoking", C = "Lung\ncancer", B = "Bronchitis", X = "Positive\nX-ray", D = "Dyspnoea")
edges <- list(c("S","C"), c("S","B"), c("C","X"), c("C","D"), c("B","D"))
plot(NA, xlim = c(-0.05, 1.05), ylim = c(0, 1.02), axes = FALSE, xlab = "", ylab = "",
     main = "Bayesian network for a chest clinic (reduced Lauritzen-Spiegelhalter 'Asia')")
for (e in edges) { a <- pos[[e[1]]]; b <- pos[[e[2]]]; d <- b - a; u <- d / sqrt(sum(d^2))
  arrows(a[1] + 0.09 * u[1], a[2] + 0.09 * u[2], b[1] - 0.09 * u[1], b[2] - 0.09 * u[2], length = 0.12, col = GREY, lwd = 1.6) }
for (k in names(pos)) { p <- pos[[k]]
  symbols(p[1], p[2], circles = 0.085, inches = FALSE, add = TRUE, bg = ifelse(k %in% c("S","C","B"), "#dfe7f1", "#fbe3d3"), fg = BLUE)
  text(p[1], p[2], lab[[k]], cex = 0.8) }
text(1.0, 0.98, "blue: causes / risk factors\norange: observable findings", cex = 0.72, col = GREY, adj = 1)
cpt <- list(
  "P(S=yes)"        = c(`(no parents)` = 0.50),
  "P(C=yes | S)"    = c(`S=no` = 0.01, `S=yes` = 0.10),
  "P(B=yes | S)"    = c(`S=no` = 0.30, `S=yes` = 0.60),
  "P(X=yes | C)"    = c(`C=no` = 0.05, `C=yes` = 0.98),
  "P(D=yes | C,B)"  = c(`C=no,B=no` = 0.10, `C=no,B=yes` = 0.80, `C=yes,B=no` = 0.70, `C=yes,B=yes` = 0.90))
for (nm in names(cpt)) { cat(nm, ":\n"); print(cpt[[nm]]); cat("\n") }
cat(sprintf("free parameters in the factorised model: %d  (versus 2^5 - 1 = 31 for the unrestricted joint)\n",
            sum(sapply(cpt, length))))



<p class="caption"><b>Figure 6.</b> The network. Smoking is a risk factor for both lung cancer and bronchitis; lung cancer produces an abnormal chest X-ray and dyspnoea; bronchitis produces dyspnoea but not an abnormal X-ray. There is no arrow from Smoking to X-ray: the graph asserts that, once the presence or absence of cancer is known, smoking history carries no further information about the X-ray. Every such missing arrow is an assumption of conditional independence.</p>

The factorisation implied by Figure 6 is \(P(S, C, B, X, D) = P(S)\, P(C \mid S)\, P(B \mid S)\, P(X \mid C)\, P(D \mid C, B)\). Counting parameters shows what the graph buys. The full joint distribution of five binary variables has \(2^5 - 1 = 31\) free probabilities; the factorised form needs one for \(P(S)\), two each for \(P(C \mid S)\), \(P(B \mid S)\) and \(P(X \mid C)\), and four for \(P(D \mid C, B)\): eleven in total, each a quantity a clinician can state or a registry can estimate.

### 5.2 Inference by enumeration, traced by hand

Inference in a Bayesian network means computing the posterior distribution of some variables (the <em>query</em>) given observed values of others (the <em>evidence</em>), with the remaining variables unobserved. The simplest exact algorithm is **enumeration**: write out the joint probability of every complete assignment using the factorisation, keep the rows consistent with the evidence, and sum over the unobserved variables:

\[ P(C \mid D = \text{yes}, X = \text{yes}) = \frac{\sum_{S, B} P(S, C, B, X = \text{yes}, D = \text{yes})}{\sum_{S, C, B} P(S, C, B, X = \text{yes}, D = \text{yes})}. \]

**By hand, the prior of cancer.** Before any evidence, \(P(C = \text{yes}) = \sum_S P(C = \text{yes} \mid S)\,P(S) = 0.10 \times 0.5 + 0.01 \times 0.5 = 0.055\). So in this population 5.5 % have lung cancer. **With dyspnoea observed**, the sum runs over the four hidden configurations of \((S, B)\) weighted by the factorised joint; the numerator keeps \(C = \text{yes}\), the denominator both values of \(C\), and the ratio comes to about 0.10 — a modest rise, because bronchitis explains dyspnoea equally well. **Adding a positive X-ray** multiplies each surviving row by \(P(X = \text{yes} \mid C)\), which is 0.98 for \(C = \text{yes}\) and only 0.05 for \(C = \text{no}\); this tilts the ratio sharply toward cancer, to about 0.69. The chunk builds the 32-row joint from the CPTs and a general query function, and confirms each number.



In [ ]:
# The full joint distribution of the network (32 rows), built from the CPTs, and a query function by enumeration.
pS <- 0.50; pC <- c(0.01, 0.10); pB <- c(0.30, 0.60); pX <- c(0.05, 0.98)       # indexed by parent value + 1
pD <- c("0,0" = 0.10, "0,1" = 0.80, "1,0" = 0.70, "1,1" = 0.90)                 # keyed by (C, B)
bern <- function(p, v) ifelse(v == 1, p, 1 - p)
jt <- expand.grid(S = 0:1, C = 0:1, B = 0:1, X = 0:1, D = 0:1)
jt$p <- with(jt, bern(pS, S) * bern(pC[S + 1], C) * bern(pB[S + 1], B) *
                 bern(pX[C + 1], X) * bern(pD[paste(C, B, sep = ",")], D))
cat(sprintf("rows: %d | probabilities sum to %.6f\n", nrow(jt), sum(jt$p)))
query <- function(target, ev = list()) {                                        # P(target = yes | evidence)
  sub <- jt; for (v in names(ev)) sub <- sub[sub[[v]] == ev[[v]], ]
  sum(sub$p[sub[[target]] == 1]) / sum(sub$p)
}
cat(sprintf("P(C = yes)                    = %.4f   (prior probability of lung cancer)\n", query("C")))
cat(sprintf("P(C = yes | D = yes)          = %.4f\n", query("C", list(D = 1))))
cat(sprintf("P(C = yes | D = yes, X = yes) = %.4f\n", query("C", list(D = 1, X = 1))))
print(head(jt[order(-jt$p), ], 4), row.names = FALSE)



The prior probability of lung cancer is 0.055, dyspnoea raises it to about 0.10, and a positive X-ray in addition raises it to about 0.69, exactly as the hand calculation predicted. The most probable complete states (table) are, unsurprisingly, the healthy ones. The next chunk adds pieces of evidence one at a time in a clinically natural order and watches the posterior move, including the effect of learning that the patient does <em>not</em> smoke, and that bronchitis is present.



In [ ]:
# How the posterior of lung cancer moves as evidence accumulates (and as alternative explanations appear).
scen <- list("no evidence" = list(), "dyspnoea" = list(D = 1), "dyspnoea + positive X-ray" = list(D = 1, X = 1),
             "dyspnoea + X-ray + smoker" = list(D = 1, X = 1, S = 1), "dyspnoea + X-ray + non-smoker" = list(D = 1, X = 1, S = 0),
             "dyspnoea + bronchitis known" = list(D = 1, B = 1), "positive X-ray only" = list(X = 1),
             "negative X-ray + dyspnoea" = list(X = 0, D = 1))
pv <- sapply(scen, function(e) query("C", e))
par(mar = c(4, 13, 2, 1))
bp <- barplot(rev(pv), horiz = TRUE, las = 1, xlim = c(0, 1), col = ifelse(rev(pv) > 0.3, ORANGE, BLUE),
              xlab = "P(lung cancer = yes | evidence)", main = "Posterior probability of lung cancer under different evidence")
text(rev(pv) + 0.02, bp, sprintf("%.3f", rev(pv)), adj = 0, cex = 0.8)



<p class="caption"><b>Figure 7.</b> Posterior probability of lung cancer under eight evidence sets. Each piece of evidence moves the posterior through the arrows of the graph: dyspnoea and the X-ray are effects of cancer and raise it; smoking is a cause and raises it further; non-smoking lowers it. Two rows deserve attention. Knowing that bronchitis is present <em>lowers</em> the posterior of cancer given dyspnoea (compare "dyspnoea" and "dyspnoea + bronchitis known"): bronchitis <em>explains away</em> the dyspnoea, so less of it remains to be explained by cancer, even though cancer and bronchitis are not directly connected. And a negative X-ray with dyspnoea leaves cancer very unlikely, because the X-ray has a sensitivity of 0.98 for cancer in this network.</p>

### 5.3 d-separation, and learning a network from data

Which variables are independent of which, given what, can be read from the graph without any arithmetic, by a criterion Pearl called **d-separation**.<sup class="c"><a href="#r10">[10]</a></sup> Three local structures account for everything:

- **Chain** \(S \rightarrow C \rightarrow X\): smoking influences the X-ray only through cancer. \(S\) and \(X\) are dependent, but become independent once \(C\) is observed — the chain is <em>blocked</em> by conditioning on the middle node.
- **Fork** (common cause) \(C \leftarrow S \rightarrow B\): cancer and bronchitis are associated because both are caused by smoking, and become independent once smoking is known. Conditioning on the common cause blocks the fork.
- **Collider** (common effect) \(C \rightarrow D \leftarrow B\): cancer and bronchitis are independent (given smoking) until the common effect, dyspnoea, is observed; conditioning on a collider <em>opens</em> the path and makes the two causes dependent — the mechanism of explaining away.

Two variables are d-separated, and hence conditionally independent, when every path between them is blocked. The chunk verifies the three statements numerically, and then shows **parameter learning** in its simplest form: it draws a synthetic registry of 5,000 patients from the network by forward sampling (each variable sampled after its parents) and recovers two CPTs by counting.



In [ ]:
# Numerical check of the three d-separation statements, then parameter learning by counting.
cat(sprintf("Chain  S->C->X : P(X|C)=%.3f  P(X|C,S=yes)=%.3f  P(X|C,S=no)=%.3f  (equal: blocked by C)\n",
            query("X", list(C = 1)), query("X", list(C = 1, S = 1)), query("X", list(C = 1, S = 0))))
cat(sprintf("Fork   C<-S->B : P(B)=%.3f vs P(B|C=yes)=%.3f (dependent); given S: P(B|S)=%.3f vs P(B|S,C)=%.3f (independent)\n",
            query("B"), query("B", list(C = 1)), query("B", list(S = 1)), query("B", list(S = 1, C = 1))))
cat(sprintf("Collider C->D<-B: given S, P(C|S)=%.3f vs P(C|S,B)=%.3f (indep); given S,D: P(C|S,D)=%.3f vs P(C|S,D,B)=%.3f (dep)\n",
            query("C", list(S = 1)), query("C", list(S = 1, B = 1)), query("C", list(S = 1, D = 1)), query("C", list(S = 1, D = 1, B = 1))))
set.seed(0); N <- 5000
S <- runif(N) < pS; C <- runif(N) < ifelse(S, pC[2], pC[1]); B <- runif(N) < ifelse(S, pB[2], pB[1])
Xr <- runif(N) < ifelse(C, pX[2], pX[1]); D <- runif(N) < pD[paste(as.integer(C), as.integer(B), sep = ",")]
reg <- data.frame(S, C, B, D)
estC <- tapply(reg$C, reg$S, mean)
cat(sprintf("\nparameter learning by counting:  estimated P(C=yes | S=no)=%.3f (true 0.010), P(C=yes | S=yes)=%.3f (true 0.100)\n",
            estC["FALSE"], estC["TRUE"]))



The numbers confirm the graph: the X-ray probability given cancer is the same whether or not the patient smokes (chain blocked); bronchitis is associated with cancer marginally but not once smoking is fixed (fork blocked); and cancer and bronchitis are independent given smoking until dyspnoea is observed, whereupon knowing bronchitis lowers the probability of cancer (collider opened). The estimated CPTs track the true values; their precision depends on the number of cases in each parent configuration, so a node with many parents needs a large registry or a prior — the general lesson of parameter learning.<sup class="c"><a href="#r13">[13]</a></sup> Both parameters and structure can be learned from data: Friedman, Geiger and Goldszmidt's <em>tree-augmented naive Bayes</em> lets each feature take one extra parent and improves on naive Bayes on most benchmarks,<sup class="c"><a href="#r14">[14]</a></sup> and structure search over the space of DAGs, guided by expert constraints, underlies the biomedical networks that Lucas and colleagues review.<sup class="c"><a href="#r15">[15]</a></sup> The practical importance of the collider case is large: selecting patients on a common effect — admission to hospital, a positive screen — induces associations between its causes that do not exist in the population, the bias epidemiology calls collider or Berkson's bias.

<div class="box bridge"><h4>Bridge to Unit 1 and to bioinformatics</h4><p>A Bayesian network is the modern form of the "reasoning with uncertainty" that Unit 1 attributed to symbolic AI: the graph is knowledge, written by hand or learned, and inference is deduction with probabilities. The same formalism underlies the hidden Markov models used for gene finding and profile alignment in BGM324 — an HMM is a Bayesian network unrolled over the positions of a sequence — and the pedigree analyses of genetic linkage, where the DAG is the family tree.</p></div>

<div class="exercise"><h4>Exercise 2</h4><p>Add the missing node of the original Asia network: a visit to Asia (A, prior 0.01) raises the probability of tuberculosis (T: 0.05 if A, 0.01 otherwise), and either tuberculosis or cancer produces the positive X-ray and the dyspnoea (replace C by "T or C" in those two CPTs). Rebuild the joint table (now 128 rows) and recompute \(P(C \mid D = \text{yes}, X = \text{yes})\). Why does it fall?</p></div>

## 6. k-Nearest Neighbours

### 6.1 The rule, by hand

The **k-nearest-neighbours** (k-NN) rule is the simplest memory-based classifier, and the archetype of what the literature calls a **lazy learner**: it builds no explicit model at training time — training consists of storing the labelled sample — and defers all computation to the moment a prediction is requested. To classify a new case \(\mathbf{x}\): compute its distance to every stored case, select the \(k\) closest, and assign the **majority class** among them; for regression, predict the **mean** of their targets. The rule was proposed in a 1951 technical report by Fix and Hodges, which remained unpublished until 1989,<sup class="c"><a href="#r16">[16]</a></sup> and its key theoretical property was established by Cover and Hart in 1967: as the training sample grows without bound, the error of the 1-nearest-neighbour rule is at most twice the Bayes error, for any distribution of the data.<sup class="c"><a href="#r17">[17]</a></sup> Half of the information needed for optimal classification is contained in the single nearest neighbour — a remarkable statement for a rule with no model, no training and one free parameter. The function used here, <code>class::knn</code>, is the implementation from Venables and Ripley.<sup class="c"><a href="#r38">[38]</a></sup>

**Worked example.** With the six training cases of Section 4.3 and the new case \(\mathbf{x} = (15, 4)\), the Euclidean distances are:

| case | features | class | \(d^2 = (15 - x_1)^2 + (4 - x_2)^2\) | \(d\) |
|---|---|---|---|---|
| 1 | (12, 2) | benign | \(9 + 4 = 13\) | 3.61 |
| 2 | (13, 3) | benign | \(4 + 1 = 5\) | 2.24 |
| 3 | (11, 1) | benign | \(16 + 9 = 25\) | 5.00 |
| 4 | (19, 9) | malignant | \(16 + 25 = 41\) | 6.40 |
| 5 | (21, 11) | malignant | \(36 + 49 = 85\) | 9.22 |
| 6 | (17, 7) | malignant | \(4 + 9 = 13\) | 3.61 |

For \(k = 1\) the nearest case is case 2 (benign), so the prediction is **benign**. For \(k = 3\) the three nearest are case 2 (\(d = 2.24\)) and then cases 1 and 6, tied at \(d = 3.61\); the three neighbours include two benign cases, and the majority is again **benign**. The distance-weighted variant of Section 6.6, in which each neighbour votes with weight \(1/d\), gives benign \(1/2.24 + 1/3.61 = 0.723\) against malignant \(1/3.61 = 0.277\) — benign by a wider margin. The three classifiers of this unit now disagree about this one point: nearest centroid and k-NN say benign, Gaussian naive Bayes says malignant, and the point lies in the overlap where no method can be sure. The chunk confirms the table.



In [ ]:
# k-NN by hand on the six cases: distances, the k nearest, the vote, confirmed with class::knn.
d <- sqrt(rowSums(sweep(six, 2, newx)^2))
tab <- data.frame(case = 1:6, x1 = six[, 1], x2 = six[, 2], class = as.character(cls), distance = round(d, 3))
print(tab[order(tab$distance), ], row.names = FALSE)
for (k in c(1, 3)) {
  o <- order(d)[1:k]; votes <- table(cls[o])
  cat(sprintf("k=%d: neighbours = cases %s, votes (benign,malignant)=(%d,%d), prediction = %s\n",
              k, paste(order(d)[1:k], collapse = ","), votes["benign"], votes["malignant"],
              as.character(knn(six, matrix(newx, 1), cls, k = k))))
}
o3 <- order(d)[1:3]; w <- 1 / d[o3]
cat(sprintf("k=3, weights 1/d: P(benign) = %.3f\n", sum(w[cls[o3] == "benign"]) / sum(w)))



<code>class::knn</code> reproduces the table and the votes. The weighted vote gives \(P(\text{benign}) = 0.723\). Ties in the vote itself (possible when \(k\) is even) are broken at random by <code>class::knn</code>; using an odd \(k\) for binary problems avoids them. The vote depends on \(k\), and the dependence can flip the answer. The lecture slides illustrate this with a configuration the chunk reproduces: a new point whose three nearest neighbours are two triangles and one square, and whose five nearest are two triangles and three squares. With \(k = 3\) the point is a triangle; with \(k = 5\) a square. Nothing about the data changed — only the size of the neighbourhood consulted.



In [ ]:
# The classic toy example: the same new point is a 'triangle' with k=3 and a 'square' with k=5.
toy_X <- rbind(c(0.6,0.8), c(-0.7,0.7), c(1.0,-0.5), c(-1.2,-1.0), c(1.3,1.1),
               c(-2.4,1.8), c(2.6,-1.9), c(-2.0,-2.5), c(2.8,2.4), c(0.3,3.0))
toy_y <- factor(c("tri","tri","sq","sq","sq","tri","sq","tri","sq","tri"))
dq <- sqrt(rowSums(toy_X^2)); ord <- order(dq)
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
for (k in c(3, 5)) {
  v <- table(toy_y[ord[1:k]]); pred <- names(which.max(v))
  cat(sprintf("k = %d: votes triangles %d / squares %d  ->  %s\n", k, v["tri"], v["sq"], pred))
  plot(toy_X, type = "n", xlim = c(-3.2, 3.4), ylim = c(-3.2, 3.4), asp = 1, xlab = "", ylab = "",
       main = sprintf("k = %d: %d triangles, %d squares -> %s", k, v["tri"], v["sq"], pred))
  symbols(0, 0, circles = dq[ord[k]] * 1.02, inches = FALSE, add = TRUE, fg = GREY, lty = 2)
  points(toy_X[toy_y == "tri", ], pch = 17, col = BLUE, cex = 1.6)
  points(toy_X[toy_y == "sq", ], pch = 15, col = ORANGE, cex = 1.4)
  points(0, 0, pch = 8, col = GREEN, cex = 2, lwd = 2)
}



<p class="caption"><b>Figure 8.</b> The same ten stored cases and the same new point (star), with the neighbourhood of the \(k\) nearest cases drawn as a dashed circle. With \(k = 3\) the circle contains two triangles and one square and the point is classified as a triangle; with \(k = 5\) it contains two triangles and three squares and the point becomes a square. The flip is not a defect of the rule but a reminder that \(k\) is a modelling choice: Section 6.4 makes it by validation rather than by inspection.</p>

### 6.2 What "nearest" means: distance metrics

The rule depends entirely on the definition of distance. For two feature vectors \(\mathbf{a}\) and \(\mathbf{b}\) in \(\mathbb{R}^p\), the **Minkowski distance** of order \(q\) is

\[ d_q(\mathbf{a}, \mathbf{b}) = \Big( \sum_{j=1}^{p} |a_j - b_j|^q \Big)^{1/q}, \]

of which the **Euclidean** distance (\(q = 2\)) and the **Manhattan** or city-block distance (\(q = 1\)) are the common cases; as \(q \to \infty\) the distance tends to the largest single coordinate difference (Chebyshev). Euclidean distance is the default and suits continuous features on consistent scales; Manhattan is less dominated by a single large deviation and is often preferred in high dimensions.<sup class="c"><a href="#r21">[21]</a></sup> For binary features the Hamming distance (number of mismatches) is the analogue, and for expression profiles the cosine or correlation distance, which ignores overall magnitude. The chunk computes the distances between two real cases and draws the set of points at unit distance from the origin under each metric — the shape a neighbourhood takes.



In [ ]:
# Three Minkowski distances between the same two cases, and the unit 'circle' of each metric.
a <- as.numeric(tr[1, feat2]); b <- as.numeric(tr[2, feat2])
cat(sprintf("case A = (%.0f,%.0f), case B = (%.0f,%.0f)\n", a[1], a[2], b[1], b[2]))
for (q in list(c(1, "Manhattan"), c(2, "Euclidean"), c(3, "Minkowski q=3"), c(Inf, "Chebyshev"))) {
  qq <- as.numeric(q[1]); dd <- if (is.infinite(qq)) max(abs(a - b)) else sum(abs(a - b)^qq)^(1 / qq)
  cat(sprintf("  %-14s distance = %.3f\n", q[2], dd))
}
th <- seq(0, 2 * pi, length.out = 400); u <- cbind(cos(th), sin(th))
plot(NA, xlim = c(-1.4, 1.4), ylim = c(-1.4, 1.4), asp = 1, xlab = "a1 - b1", ylab = "a2 - b2",
     main = "Points at distance 1 from the origin")
for (q in list(c(1, ORANGE), c(2, BLUE), c(Inf, GREEN))) {
  qq <- as.numeric(q[1])
  nrm <- if (is.infinite(qq)) apply(abs(u), 1, max) else rowSums(abs(u)^qq)^(1 / qq)
  lines(u / nrm, col = q[2], lwd = 2)
}
legend("topright", c("Manhattan (q=1)", "Euclidean (q=2)", "Chebyshev (q->inf)"),
       col = c(ORANGE, BLUE, GREEN), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 9.</b> The set of points at unit distance from the centre under three metrics: a diamond for Manhattan, a circle for Euclidean, a square for Chebyshev. A k-NN neighbourhood has the shape of the metric's unit ball, so the choice of metric decides which cases count as neighbours. The printed distances between two real cases rank in the expected order, \(d_1 \geq d_2 \geq d_\infty\).</p>

### 6.3 Feature scaling is not optional

Every Minkowski distance adds differences across features as if they were commensurable. In the <code>mlbench</code> breast-cancer data they happen to be — all nine features are integer scores from 1 to 10 — so standardising them changes little, and this is one place where the R dataset is gentler than scikit-learn's, whose features span "mean area" in the hundreds and "fractal dimension" in the second decimal. To exhibit the lesson that matters in general, the chunk <em>simulates</em> a mis-scaled feature by multiplying one feature (Mitoses) by 100, as if it had been recorded in different units, and shows that k-NN on the raw features then becomes a one-feature classifier, which standardisation repairs.



In [ ]:
# Feature scaling: on the native (comparable) data it barely matters; on a mis-scaled copy it is decisive.
knn_cv <- function(X, y, k = 5, Kf = 5, standardize = FALSE) {
  set.seed(1); folds <- sample(rep(1:Kf, length.out = nrow(X)))
  mean(sapply(1:Kf, function(f) {
    trI <- folds != f; Xtr <- X[trI, , drop = FALSE]; Xte <- X[!trI, , drop = FALSE]
    if (standardize) { ct <- colMeans(Xtr); s <- apply(Xtr, 2, sd); Xtr <- scale(Xtr, ct, s); Xte <- scale(Xte, ct, s) }
    mean(knn(Xtr, Xte, y[trI], k = k) == y[!trI]) }))
}
Xnat <- as.matrix(bc[, 1:9]); set.seed(3)
Xaug <- cbind(Xnat, noise = rnorm(nrow(Xnat)) * 500)        # one feature recorded on a large scale (here pure noise)
cat(sprintf("native features (all 1-10):         raw %.3f  |  standardised %.3f\n",
            knn_cv(Xnat, bc$Class), knn_cv(Xnat, bc$Class, standardize = TRUE)))
cat(sprintf("plus one large-scale noise feature:  raw %.3f  |  standardised %.3f\n",
            knn_cv(Xaug, bc$Class), knn_cv(Xaug, bc$Class, standardize = TRUE)))



On the native data, where every feature already lives on the 1–10 scale, standardising barely moves the cross-validated accuracy. Add a single feature on a large scale — here pure noise multiplied by 500 — and raw k-NN collapses (to about 0.78), because the Euclidean distance is then dominated by that one feature and the nine informative ones become numerically invisible; standardising brings every feature to unit variance and restores the accuracy. The same applies to the support vector machine of Section 7 and to every method based on distances or inner products; naive Bayes and decision trees (Unit 4) are the exceptions, since they treat each feature on its own scale.

<div class="box warn"><h4>Common misconception</h4><p>"Standardising the whole dataset before splitting is harmless." The mean and standard deviation of the whole dataset include the test cases, so the test cases have leaked into the training procedure. The leak is small for scaling and large for other preprocessing steps (feature selection, imputation); the habit of fitting the scaler on the training fold only — as the <code>knn_cv</code> helper does — avoids it uniformly. Supplement S1 treats leakage in detail.</p></div>

### 6.4 Choosing k

The parameter \(k\) controls the smoothness of the rule. With \(k = 1\) every training case is classified correctly — its nearest neighbour is itself — and the boundary wraps around every isolated case, including mislabelled ones: the rule has memorised the sample, which is overfitting in its purest form. As \(k\) grows the vote averages over a larger neighbourhood, the boundary smooths, and the influence of any single case fades; in the limit \(k = n\) every point is assigned the majority class of the whole sample. The right \(k\) balances these effects and is chosen by validation, never by the training error. The chunk computes a **validation curve**: training accuracy and 5-fold cross-validated accuracy against \(k\), on the standardised nine features.



In [ ]:
# Validation curve for k: training accuracy versus cross-validated accuracy (standardised nine features).
Xs <- scale(Xnat); ks <- seq(1, 59, by = 2)
tr_acc <- sapply(ks, function(k) mean(knn(Xs, Xs, bc$Class, k = k) == bc$Class))
cv_a   <- sapply(ks, function(k) knn_cv(Xnat, bc$Class, k = k, standardize = TRUE))
best_k <- ks[which.max(cv_a)]
cat(sprintf("k=1: training accuracy %.3f, CV accuracy %.3f\n", tr_acc[1], cv_a[1]))
cat(sprintf("best k by CV: %d (CV accuracy %.3f); CV accuracy at k=59: %.3f\n", best_k, max(cv_a), cv_a[length(ks)]))
plot(ks, tr_acc, type = "b", pch = 19, cex = 0.5, col = GREY, ylim = c(0.93, 1.002),
     xlab = "k (number of neighbours)", ylab = "accuracy", main = "Validation curve of k-NN on the breast-cancer data")
lines(ks, cv_a, type = "b", pch = 19, cex = 0.5, col = BLUE)
abline(v = best_k, col = ORANGE, lty = 2)
legend("bottomright", c("training accuracy", "5-fold CV accuracy", sprintf("best k = %d", best_k)),
       col = c(GREY, BLUE, ORANGE), lty = c(1, 1, 2), pch = c(19, 19, NA), bty = "n", cex = 0.8)



<p class="caption"><b>Figure 10.</b> Training accuracy (grey) and cross-validated accuracy (blue) as \(k\) runs from 1 to 59. At \(k = 1\) the training accuracy is 1.000 by construction while the cross-validated accuracy is lower; the gap is the overfitting. The cross-validated curve rises to a broad maximum and then declines slowly as the neighbourhood becomes too large to follow the boundary. The maximum is broad: a range of \(k\) is defensible, and the single "best" value should not be over-interpreted (Supplement S1 discusses the one-standard-error rule).</p>

The effect of \(k\) on the shape of the boundary is best seen in two dimensions. The next chunk fits k-NN with \(k = 1\), 5 and 25 to the standardised two-feature problem and draws the regions.



In [ ]:
# Decision regions of k-NN for k = 1, 5 and 25 on the two-feature problem (standardised).
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (k in c(1, 5, 25)) {
  predk <- function(G) knn(sc(tr), scale(G, center = ctr, scale = csd), tr$Class, k = k) == "malignant"
  tra <- mean(knn(sc(tr), sc(tr), tr$Class, k = k) == tr$Class); tea <- mean(knn(sc(tr), sc(te), tr$Class, k = k) == te$Class)
  shade_regions(predk, xlim, ylim, cn = feat2, res = 120,
                main = sprintf("k = %d: train %.3f, test %.3f", k, tra, tea),
                xlab = "clump thickness", ylab = if (k == 1) "cell size" else "")
  points(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.4)
}



<p class="caption"><b>Figure 11.</b> Decision regions for \(k = 1\), 5 and 25, with the training cases. With \(k = 1\) the boundary is a jagged frontier with islands around every isolated case (training accuracy 1.000, and the islands are noise); with \(k = 5\) most islands disappear; with \(k = 25\) the boundary is smooth and the few remaining training errors are cases deep inside the other class's territory. The test accuracies, printed in the titles, are close for \(k = 5\) and 25 and lower for \(k = 1\).</p>

### 6.5 The curse of dimensionality

The k-NN rule assumes that near neighbours exist. In high-dimensional spaces they may not, in a sense Richard Bellman named the <em>curse of dimensionality</em> in 1961<sup class="c"><a href="#r20">[20]</a></sup> and that Beyer and colleagues made specific in 1999: as the dimension \(p\) grows, the distance from a query point to its nearest neighbour approaches the distance to its farthest, so the contrast between "near" and "far" vanishes.<sup class="c"><a href="#r21">[21]</a></sup> To enclose 10 % of points uniformly distributed in the unit cube, a cube of side \(0.1^{1/p}\) is needed — 0.32 for \(p = 2\), 0.63 for \(p = 5\), 0.79 for \(p = 10\): the "neighbourhood" spans most of each axis. The chunk simulates the concentration of distances.



In [ ]:
# Distance concentration: ratio of farthest to nearest neighbour distance as the dimension grows (uniform data).
set.seed(0); dims <- c(1, 2, 3, 5, 10, 20, 50, 100, 200, 500, 1000); npts <- 500; nq <- 50
ratio <- sapply(dims, function(p) {
  pts <- matrix(runif(npts * p), npts); qs <- matrix(runif(nq * p), nq)
  median(apply(qs, 1, function(q) { dd <- sqrt(colSums((t(pts) - q)^2)); max(dd) / min(dd) })) })
summ <- data.frame(p = dims, farthest_over_nearest = round(ratio, 2), side_for_10pct = round(0.1^(1 / dims), 2))
print(summ, row.names = FALSE)
plot(dims, ratio, type = "b", pch = 19, col = BLUE, log = "xy", xlab = "number of dimensions p",
     ylab = "farthest / nearest (median)", main = "Distance concentration: 'nearest' loses meaning as p grows")



<p class="caption"><b>Figure 12.</b> Median ratio of the farthest to the nearest distance among 500 uniform points, against the dimension. In one dimension the farthest point is hundreds of times farther than the nearest; by \(p = 1000\) it is barely 10 % farther (printed table). The last column gives the side of the cube needed to enclose 10 % of the data. Real data lie near lower-dimensional structures, which is why k-NN works at all with nine or sixty-four features — but with thousands of features (gene expression, spectra) k-NN needs feature selection or a learned distance.</p>

### 6.6 Weighted k-NN, regression and cost

Two extensions are routine. **Distance weighting** lets each neighbour vote with a weight that decreases with distance, typically \(1/d\), so a close neighbour counts more; Dudani introduced it in 1976 and it reduces sensitivity to the exact \(k\).<sup class="c"><a href="#r18">[18]</a></sup> **k-NN regression** predicts a continuous target by the (weighted) mean of the neighbours' targets; it is the simplest non-parametric regression, and Altman's exposition is the standard introduction.<sup class="c"><a href="#r19">[19]</a></sup> Because R has no built-in diabetes dataset, the chunk applies both to <code>BostonHousing</code> — 506 suburbs, 13 predictors and a continuous median house value, the R stand-in for the Python course's diabetes regression<sup class="c"><a href="#r35">[35]</a></sup> — and compares the cross-validated \(R^2\) with linear regression; it also times prediction against "training" to show where k-NN spends its effort.



In [ ]:
# k-NN regression on BostonHousing (uniform and distance weights) vs linear regression; and the cost profile.
data(BostonHousing, package = "mlbench")
Xb <- scale(data.matrix(BostonHousing[, 1:13])); yb <- BostonHousing$medv   # data.matrix: chas factor -> numeric
knn_reg <- function(Xtr, Xte, ytr, k, weights = "uniform") {
  D <- as.matrix(dist(rbind(Xte, Xtr)))[1:nrow(Xte), -(1:nrow(Xte))]
  apply(D, 1, function(dr) { o <- order(dr)[1:k]; w <- if (weights == "distance") 1 / pmax(dr[o], 1e-9) else rep(1, k)
                             sum(w * ytr[o]) / sum(w) }) }
set.seed(1); folds <- sample(rep(1:5, length.out = nrow(Xb)))
cvR2 <- function(k, w) { pr <- numeric(nrow(Xb))
  for (f in 1:5) { trI <- folds != f; pr[!trI] <- knn_reg(Xb[trI, ], Xb[!trI, ], yb[trI], k, w) }
  1 - sum((yb - pr)^2) / sum((yb - mean(yb))^2) }
res <- sapply(c("uniform", "distance"), function(w) sapply(c(1, 5, 15, 30), function(k) round(cvR2(k, w), 3)))
rownames(res) <- paste0("k=", c(1, 5, 15, 30)); print(res)
linR2 <- { pr <- numeric(nrow(Xb)); df <- BostonHousing
  for (f in 1:5) { trI <- folds != f; pr[!trI] <- predict(lm(medv ~ ., df[trI, ]), df[!trI, ]) }
  1 - sum((yb - pr)^2) / sum((yb - mean(yb))^2) }
cat(sprintf("linear regression (Unit 2 method): CV R2 = %.3f\n", linR2))
big <- scale(matrix(rnorm(4000 * 20), 4000)); ybig <- factor(sample(c("a", "b"), 4000, TRUE))
t_fit <- system.time(invisible(NULL))["elapsed"]                                   # k-NN 'training' = storing: free
t_pred <- system.time(knn(big, big[1:800, ], ybig, k = 5))["elapsed"]
cat(sprintf("cost: storing 4000x20 training cases is free; predicting 800 queries took %.3f s (all distances computed)\n", t_pred))



The pattern echoes classification: an intermediate \(k\) is best (around \(k = 5\)), very large \(k\) smooths toward the global mean, and distance weighting helps most — it lifts the best cross-validated \(R^2\) to about 0.79. Here k-NN <em>edges out</em> the linear model (\(R^2 \approx 0.72\)), the opposite of the Python course's diabetes result: median house values depend non-linearly on the predictors, and on such data a method that assumes nothing can beat one that assumes linearity, provided the sample is dense enough to find near neighbours. The lesson is that which family wins is a property of the data, not of the method. The timing shows k-NN's inverted cost profile — training is free, every prediction computes all \(n\) distances — the reverse of every other classifier in this course. Space-partitioning structures (the k-d tree of Bentley,<sup class="c"><a href="#r22">[22]</a> the ball tree) speed the search in low dimensions but stop helping beyond a few dozen, a consequence of Section 6.5. Memory-based methods remain valuable in biomedicine where the sample is itself the object of interest — k-NN imputation of missing values in expression matrices is a standard preprocessing step.<sup class="c"><a href="#r23">[23]</a></sup>

<div class="exercise"><h4>Exercise 3</h4><p>In the validation-curve chunk of Section 6.4, change the distance to Manhattan by replacing the Euclidean step of <code>knn</code> with an explicit Manhattan k-NN (or scale the features differently). Does the best \(k\) change? Then add distance weighting to the six-case example and explain why the training accuracy of a distance-weighted 1-NN must still be 1.000.</p></div>

## 7. Support Vector Machines

### 7.1 The maximum-margin idea

Return to the geometric view. A support vector machine is a <strong>discriminative classifier that outputs an optimal hyperplane</strong>: given labelled training cases, it returns the hyperplane that best separates the classes, and new cases are classified by the side on which they fall. Data are **linearly separable** when such a hyperplane exists — a line in two dimensions, a plane in three — and **non-linearly separable** when a curved boundary is required (Section 7.3). When two classes are linearly separable, infinitely many hyperplanes separate them, and the perceptron of Unit 1 stops at the first it finds. Vapnik and his collaborators asked which separating hyperplane is <em>best</em>, and answered: the one farthest from the nearest training case of either class — the hyperplane with the **maximum margin**, the perpendicular distance from the hyperplane to the closest training points.<sup class="c"><a href="#r24">[24]</a>,<a href="#r26">[26]</a></sup> The argument is one of robustness: a boundary that passes close to training cases will misclassify new cases that fall slightly beyond them, whereas a wide empty corridor tolerates such perturbations. The cases that lie exactly on the edge of the corridor determine it and are called **support vectors**; moving any other case changes nothing.

Write the hyperplane as \(\mathbf{w}^\top \mathbf{x} + b = 0\) and code the classes as \(y_i \in \{-1, +1\}\). Scaling \(\mathbf{w}\) and \(b\) so that the nearest points satisfy \(|\mathbf{w}^\top \mathbf{x}_i + b| = 1\) makes the width of the corridor \(2 / \lVert \mathbf{w} \rVert\), so maximising the margin is minimising \(\lVert \mathbf{w} \rVert\):

<div class="eq">\[ \min_{\mathbf{w}, b} \; \tfrac{1}{2} \lVert \mathbf{w} \rVert^2 \qquad \text{subject to} \qquad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 \quad \text{for all } i. \]
<span class="where">This is the <strong>hard-margin</strong> support vector machine: a convex quadratic programme with a unique solution whenever the data are separable. In the solution \(\mathbf{w} = \sum_i \alpha_i y_i \mathbf{x}_i\) with \(\alpha_i > 0\) only for support vectors.</span></div>

The R function <code>e1071::svm</code> is an interface to LIBSVM.<sup class="c"><a href="#r33">[33]</a>,<a href="#r37">[37]</a></sup> The chunk below generates two separable clusters, fits a hard-margin machine (a very large \(C\)), and draws the hyperplane, the margins and the support vectors.



In [ ]:
# The maximum-margin hyperplane on separable data: hyperplane, margin corridor and support vectors.
bl <- make_blobs(60, rbind(c(0, 0), c(3.5, 3.5)), sd = 0.6); Xbl <- bl$X; ybl <- bl$y
hard <- svm(Xbl, ybl, kernel = "linear", cost = 1e5, scale = FALSE)
w <- as.numeric(t(hard$coefs) %*% hard$SV); b <- -hard$rho
cat(sprintf("w = (%.3f, %.3f), b = %.3f, ||w|| = %.3f, margin width 2/||w|| = %.3f\n", w[1], w[2], b, sqrt(sum(w^2)), 2 / sqrt(sum(w^2))))
cat(sprintf("support vectors: %d of %d cases\n", hard$tot.nSV, length(ybl)))
plot(Xbl, col = ifelse(ybl == levels(ybl)[1], ORANGE, BLUE), pch = 19, cex = 0.9,
     xlab = "x1", ylab = "x2", main = "Maximum-margin hyperplane (solid) and margins (dashed)")
abline(a = -b / w[2], b = -w[1] / w[2], lwd = 2)                               # w.x + b = 0
abline(a = (-b + 1) / w[2], b = -w[1] / w[2], lty = 2); abline(a = (-b - 1) / w[2], b = -w[1] / w[2], lty = 2)
points(hard$SV, cex = 2.2, col = GREEN, lwd = 2)
legend("topleft", c("class -1", "class +1", "support vectors"), col = c(ORANGE, BLUE, GREEN),
       pch = c(19, 19, 1), bty = "n", cex = 0.8)



<p class="caption"><b>Figure 13.</b> Sixty separable points, the maximum-margin hyperplane (solid) and the two margin lines (dashed) at signed distance ± 1. The circled points are the support vectors: they lie on the margin lines and the hyperplane is determined by them alone. Every other point could be moved anywhere on its own side of the corridor without changing the solution — the solution is <em>sparse</em> in the data, the property that gives the method its name.</p>

### 7.2 The soft margin and the parameter C

Real classes overlap, and the hard-margin problem then has no solution. Cortes and Vapnik's 1995 formulation introduced **slack variables** \(\xi_i \geq 0\) that measure how far each case violates the margin, and a parameter \(C\) that prices the violations:<sup class="c"><a href="#r25">[25]</a></sup>

<div class="eq">\[ \min_{\mathbf{w}, b, \boldsymbol{\xi}} \; \tfrac{1}{2} \lVert \mathbf{w} \rVert^2 + C \sum_{i} \xi_i \qquad \text{subject to} \qquad y_i (\mathbf{w}^\top \mathbf{x}_i + b) \geq 1 - \xi_i, \quad \xi_i \geq 0. \]
<span class="where">A <strong>large \(C\)</strong> punishes every violation: fewer training cases are misclassified, the margin shrinks, the boundary bends to the data — a risk of <em>overfitting</em> (low bias, high variance). A <strong>small \(C\)</strong> tolerates violations for a larger margin: more general and more stable, at the risk of <em>underfitting</em> (high bias, low variance).</span></div>

\(C\) is the regularisation parameter of the support vector machine, the inverse of the \(\lambda\) of ridge regression in Unit 2, and like \(k\) in k-NN it is chosen by cross-validation. The chunk fits linear machines with \(C = 0.01\), 1 and 100 to the two-feature problem; <code>svm</code> standardises the features internally by default.



In [ ]:
# The effect of C on a linear soft-margin SVM (two features): margin width, support vectors, accuracy.
svm_region_bc <- function(fit, main, res = 120) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res)
  G <- setNames(expand.grid(gx, gy), feat2)
  image(gx, gy, matrix(as.integer(predict(fit, G) == "malignant"), res), col = c("#dfe7f1", "#fbe3d3"),
        main = main, xlab = "clump thickness", ylab = "cell size")
  dv <- attr(predict(fit, G, decision.values = TRUE), "decision.values")[, 1]
  contour(gx, gy, matrix(dv, res), levels = c(-1, 0, 1), add = TRUE, drawlabels = FALSE, lty = c(2, 1, 2))
  points(jitter(tr[[feat2[1]]]), jitter(tr[[feat2[2]]]), col = pcol(tr), pch = 19, cex = 0.4)
}
par(mfrow = c(1, 3), mar = c(4, 4, 3, 1))
for (C in c(0.01, 1, 100)) {
  fit <- svm(f2, data = tr, kernel = "linear", cost = C)
  svm_region_bc(fit, sprintf("C = %g: %d SVs, test %.3f", C, fit$tot.nSV, mean(predict(fit, te) == te$Class)))
}



<p class="caption"><b>Figure 14.</b> Linear soft-margin machines with \(C = 0.01\), 1 and 100; solid line, the boundary; dashed lines, the margins. With \(C = 0.01\) the margin is enormous and almost every case is a support vector; with \(C = 1\) and 100 the corridor is narrow, only the cases in the overlap support it, and the two solutions are almost identical, because once \(C\) is large enough that violations are expensive, the same few overlapping cases determine the boundary. The test accuracies differ little on this easy problem — a lesson in itself: \(C\) matters most where the classes overlap heavily or the sample is small.</p>

### 7.3 The kernel trick

A linear boundary cannot separate classes arranged in concentric rings or interleaved crescents. The remedy that makes the support vector machine general is the observation, due to Boser, Guyon and Vapnik in 1992,<sup class="c"><a href="#r24">[24]</a></sup> that the training problem and the decision function depend on the data only through inner products \(\mathbf{x}_i^\top \mathbf{x}_j\). If the data are first mapped into a higher-dimensional space by some \(\phi(\mathbf{x})\), a linear boundary there is a curved boundary in the original space, and the inner products become \(\phi(\mathbf{x}_i)^\top \phi(\mathbf{x}_j)\). A **kernel** \(K(\mathbf{x}_i, \mathbf{x}_j)\) computes this inner product directly, without ever constructing \(\phi\); Mercer's theorem of 1909 characterises the functions for which such a \(\phi\) exists.<sup class="c"><a href="#r27">[27]</a></sup> The decision function becomes \(f(\mathbf{x}) = \sum_{i \in \text{SV}} \alpha_i y_i K(\mathbf{x}_i, \mathbf{x}) + b\), a weighted sum of similarities between the new case and the support vectors. The common kernels are the **linear** kernel, the **polynomial** kernel \((\gamma\, \mathbf{x}_i^\top \mathbf{x}_j + r)^d\), and the **radial basis function (RBF)** kernel

\[ K(\mathbf{x}_i, \mathbf{x}_j) = \exp\!\big(-\gamma\, \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2\big), \]

whose implicit feature space is infinite-dimensional. A fourth kernel, the **sigmoid** \(\tanh(\gamma\, \mathbf{x}_i^\top \mathbf{x}_j + r)\), makes the decision function resemble a two-layer neural network; it is not a valid Mercer kernel for all parameters and is rarely the best choice. Schölkopf and Smola's monograph is the standard treatment.<sup class="c"><a href="#r28">[28]</a></sup> The chunk shows the kernels on the two classical non-linear problems.



In [ ]:
# Linear, polynomial and RBF (and sigmoid) kernels on two non-linearly separable problems (moons and circles).
svm_panel <- function(fit, X, y, xlim, ylim, main, res = 120) {
  gx <- seq(xlim[1], xlim[2], length.out = res); gy <- seq(ylim[1], ylim[2], length.out = res); G <- as.matrix(expand.grid(gx, gy))
  dv <- attr(predict(fit, G, decision.values = TRUE), "decision.values")[, 1]
  image(gx, gy, matrix(as.integer(predict(fit, G) == levels(y)[2]), res), col = c("#fbe3d3", "#dfe7f1"),
        main = main, xlab = "", ylab = "", xaxt = "n", yaxt = "n")
  contour(gx, gy, matrix(dv, res), levels = 0, add = TRUE, drawlabels = FALSE, lwd = 1.2)
  points(X, col = ifelse(y == levels(y)[1], ORANGE, BLUE), pch = 19, cex = 0.4)
}
probs <- list(moons = make_moons(300, 0.25), circles = make_circles(300, 0.1, 0.4))
kerns <- list(c("linear", "linear"), c("poly, deg 3", "polynomial"), c("RBF, gamma 1", "radial"), c("sigmoid", "sigmoid"))
par(mfrow = c(2, 4), mar = c(1.5, 1.5, 3, 0.8))
for (pn in names(probs)) {
  X <- probs[[pn]]$X; y <- probs[[pn]]$y; set.seed(0); ti <- sample(nrow(X), 0.7 * nrow(X))
  xl <- range(X[, 1]) + c(-.4, .4); yl <- range(X[, 2]) + c(-.4, .4)
  for (kn in kerns) {
    fit <- svm(X[ti, ], y[ti], kernel = kn[2], cost = 1, gamma = if (kn[2] == "radial") 1 else if (kn[2] == "sigmoid") 0.5 else 1, degree = 3)
    svm_panel(fit, X[ti, ], y[ti], xl, yl, sprintf("%s, %s\ntest %.2f", pn, kn[1], mean(predict(fit, X[-ti, ]) == y[-ti])))
  }
}



<p class="caption"><b>Figure 15.</b> Two non-linear problems (rows) and four kernels (columns), with the training points and the decision boundary. The linear kernel cannot do better than a straight cut; the cubic polynomial separates the moons well but describes the rings poorly; the RBF kernel separates both to high accuracy; the sigmoid kernel produces a boundary of its own shape and does not separate either well with these parameters, which is representative of its reputation. The RBF kernel is the default of <code>svm</code> and of LIBSVM,<sup class="c"><a href="#r33">[33]</a></sup> and the usual first choice when nothing is known about the shape of the boundary.</p>

Why a non-linear map makes a non-linear problem linear can be seen without any kernel, with the "lifting" picture of the lecture slides. The rings are not linearly separable in the plane \((x_1, x_2)\). Add a third feature \(z = x_1^2 + x_2^2\) — the squared distance from the centre — and the inner disc stays low while the outer ring rises, so a threshold on \(z\) separates them. The chunk performs the lifting and fits a linear machine before and after.



In [ ]:
# Lifting a non-linearly separable problem into a space where it is linearly separable: add z = x1^2 + x2^2.
ci <- make_circles(300, 0.08, 0.4); Xc <- ci$X; yc <- ci$y; z <- rowSums(Xc^2)
lin2 <- svm(Xc, yc, kernel = "linear", cost = 1); lin3 <- svm(cbind(Xc, z), yc, kernel = "linear", cost = 1)
cat(sprintf("linear SVM training accuracy: in (x1,x2) = %.3f | in (x1,x2,z) = %.3f\n",
            mean(predict(lin2, Xc) == yc), mean(predict(lin3, cbind(Xc, z)) == yc)))
zcut <- mean(c(mean(z[yc == "inner"]), mean(z[yc == "outer"])))
par(mfrow = c(1, 2), mar = c(4, 4, 3, 1))
plot(Xc, col = ifelse(yc == "outer", ORANGE, BLUE), pch = 19, cex = 0.5, asp = 1,
     xlab = "x1", ylab = "x2", main = "Original plane: not linearly separable")
plot(Xc[, 1], z, col = ifelse(yc == "outer", ORANGE, BLUE), pch = 19, cex = 0.5,
     xlab = "x1", ylab = "z = x1^2 + x2^2", main = "Lifted feature z: a threshold separates the classes")
abline(h = zcut, col = GREEN, lwd = 2, lty = 2)
legend("topright", c("outer ring", "inner disc", "separating z"), col = c(ORANGE, BLUE, GREEN),
       pch = c(19, 19, NA), lty = c(NA, NA, 2), bty = "n", cex = 0.8)



<p class="caption"><b>Figure 16.</b> Left: the rings in their original coordinates, where a linear machine is little better than chance (training accuracy printed). Right: the same points with \(z = x_1^2 + x_2^2\) on the vertical axis; the inner disc lies at low \(z\), the outer ring at high \(z\), and the green line (a linear boundary in the lifted space) separates them — back in the original plane that line is a circle. This two-dimensional view replaces the Python unit's three-dimensional plot; the kernel trick achieves the same for maps too large to write out, by computing only inner products.</p>

### 7.4 Gamma, and the two knobs of an RBF machine

The RBF kernel has a width parameter \(\gamma\). A **small \(\gamma\)** makes each Gaussian bump wide, so distant points influence the decision and the boundary is smooth, approaching linear. A **large \(\gamma\)** makes the bumps narrow, so only nearby points matter and the boundary follows individual cases. Together \(C\) and \(\gamma\) control the complexity of an RBF machine, and both are tuned by cross-validation over a logarithmic grid — the LIBSVM practical guide recommends exactly this, preceded by scaling.<sup class="c"><a href="#r33">[33]</a></sup> The chunk draws the boundary for every combination of three values of each on the moons.



In [ ]:
# A grid of decision boundaries: C (rows) x gamma (columns) for the RBF kernel on the moons.
mo <- make_moons(300, 0.25); Xm <- mo$X; ym <- mo$y; set.seed(0); ti <- sample(nrow(Xm), 0.7 * nrow(Xm))
xl <- range(Xm[, 1]) + c(-.4, .4); yl <- range(Xm[, 2]) + c(-.4, .4)
par(mfrow = c(3, 3), mar = c(1.2, 1.2, 2.6, 0.6))
for (C in c(0.1, 1, 100)) for (g in c(0.1, 1, 50)) {
  fit <- svm(Xm[ti, ], ym[ti], kernel = "radial", cost = C, gamma = g)
  svm_panel(fit, Xm[ti, ], ym[ti], xl, yl,
            sprintf("C=%g g=%g | test %.2f, SV %d", C, g, mean(predict(fit, Xm[-ti, ]) == ym[-ti]), fit$tot.nSV))
}



<p class="caption"><b>Figure 17.</b> RBF machines on the moons for \(C \in \{0.1, 1, 100\}\) (rows) and \(\gamma \in \{0.1, 1, 50\}\) (columns); each title gives test accuracy and the number of support vectors. Across a row, increasing \(\gamma\) changes the boundary from nearly straight to a set of islands around individual cases. Down a column, increasing \(C\) narrows the margin and removes training misclassifications at the cost of a more contorted boundary. The top-left machine underfits, the bottom-right one memorises (many support vectors, lower test accuracy — the SVM equivalent of \(k = 1\) in Figure 11), and the best test accuracy lies in the interior of the grid. A machine that keeps most of its training cases as support vectors has memorised them — a useful warning sign.</p>

### 7.5 Probabilities, regression and the hinge loss

A support vector machine outputs a signed distance to the boundary, \(f(\mathbf{x})\), not a probability. Platt proposed fitting a logistic function \(P(y = 1 \mid \mathbf{x}) = 1 / (1 + \exp(A f(\mathbf{x}) + B))\) to the outputs on held-out data — **Platt scaling**;<sup class="c"><a href="#r29">[29]</a></sup> Lin, Lin and Weng gave the numerically robust fitting procedure that LIBSVM, and hence <code>e1071::svm(probability = TRUE)</code>, implements.<sup class="c"><a href="#r30">[30]</a></sup> In a large empirical comparison, Niculescu-Mizil and Caruana found that SVM scores calibrated this way give good probabilities, whereas raw naive Bayes posteriors are among the worst calibrated.<sup class="c"><a href="#r31">[31]</a></sup> The margin idea transfers to regression by inverting it: **support vector regression** (SVR) fits a function that stays within a tube of half-width \(\varepsilon\) around the targets, penalising only points outside the tube, with the same \(C\) and kernels; Smola and Schölkopf's tutorial is the reference.<sup class="c"><a href="#r32">[32]</a></sup>

Rewriting the soft-margin problem without constraints shows what the machine minimises. Since \(\xi_i = \max(0, 1 - y_i f(\mathbf{x}_i))\) at the optimum, the problem is a **hinge loss** plus an \(L_2\) penalty, \(\sum_i \max(0, 1 - y_i f(\mathbf{x}_i)) + \tfrac{1}{2C} \lVert \mathbf{w} \rVert^2\) — the same "loss plus regularisation" structure as the logistic regression of Unit 2, whose loss is \(\log(1 + e^{-y_i f(\mathbf{x}_i)})\). The chunk draws the two losses with the zero-one loss they approximate, then demonstrates Platt scaling and SVR on real data.



In [ ]:
# Platt scaling on the breast-cancer SVM; SVR on BostonHousing; then the three losses on the margin.
svm_bc <- svm(Class ~ ., data = tr, kernel = "radial", cost = 1, probability = TRUE)
pr <- predict(svm_bc, te, decision.values = TRUE, probability = TRUE)
dv <- attr(pr, "decision.values")[, 1]; pb <- attr(pr, "probabilities")[, "benign"]
pick <- order(dv)[c(1, round(length(dv) / 4), round(length(dv) / 2), round(3 * length(dv) / 4), length(dv))]
cat("Platt-scaled P(benign) vs decision value f(x), five test cases from most to least benign:\n")
print(data.frame(f_x = round(dv[pick], 2), P_benign = round(pb[pick], 3),
                 truth = as.character(te$Class[pick])), row.names = FALSE)
set.seed(1); bf <- sample(rep(1:5, length.out = nrow(BostonHousing)))
svr_r2 <- sapply(c(1, 10, 100, 1000), function(C) {
  prv <- numeric(nrow(BostonHousing))
  for (f in 1:5) { tri <- bf != f
    prv[!tri] <- predict(svm(medv ~ ., BostonHousing[tri, ], kernel = "radial", cost = C, epsilon = 0.1), BostonHousing[!tri, ]) }
  round(1 - sum((BostonHousing$medv - prv)^2) / sum((BostonHousing$medv - mean(BostonHousing$medv))^2), 3) })
cat(sprintf("SVR (RBF, epsilon = 0.1) on BostonHousing, CV R2 by C: %s\n", paste(c(1, 10, 100, 1000), svr_r2, sep = ":", collapse = "  ")))
# The hinge loss, the logistic loss and the zero-one loss, as functions of the margin.
m <- seq(-3, 3, length.out = 400)
plot(m, as.numeric(m < 0), type = "l", col = GREY, lwd = 1.6, ylim = c(0, 4),
     xlab = "margin  m = y * f(x)  (>1: outside the corridor)", ylab = "loss", main = "Three losses on the margin")
lines(m, pmax(0, 1 - m), col = ORANGE, lwd = 2)
lines(m, log(1 + exp(-m)) / log(2), col = BLUE, lwd = 2)
abline(v = c(0, 1), col = c("black", ORANGE), lty = c(1, 3), lwd = c(0.6, 1))
legend("topright", c("zero-one loss", "hinge loss (SVM)", "logistic loss"),
       col = c(GREY, ORANGE, BLUE), lwd = 2, bty = "n", cex = 0.8)



<p class="caption"><b>Figure 18.</b> Loss against the margin \(m = y f(\mathbf{x})\). The hinge loss is zero for cases outside the corridor (\(m \geq 1\)) — they exert no force on the solution, which is why only support vectors matter — and grows linearly inside it. The logistic loss is never exactly zero: every case keeps pulling the boundary a little, which is why logistic regression has no support vectors. Both are convex upper bounds on the zero-one loss. The chunk's printed output first shows the Platt-scaled probabilities and the SVR results discussed below.</p>

The Platt-scaled probabilities increase monotonically with the decision value and pass through 0.5 near \(f(\mathbf{x}) = 0\), turning a distance into a probability that can be thresholded by cost as in Section 3.2. The SVR results show a flexible regressor on the housing data: its \(R^2\) rises with \(C\) to a high value, comparable to or above the linear and k-NN results of Section 6.6. Support vector machines have been used in medicine since the late 1990s, most influentially for the classification of tumours from gene-expression profiles, where the number of features far exceeds the number of patients and a maximum-margin classifier with recursive feature elimination proved both accurate and interpretable.<sup class="c"><a href="#r34">[34]</a></sup>

<div class="box key"><h4>Key idea</h4><p>A support vector machine is defined by three choices — the kernel (shape of the boundary), \(C\) (tolerance for training violations) and, for the RBF kernel, \(\gamma\) (smoothness) — and its solution depends only on the support vectors. The method is not probabilistic, scales poorly beyond some tens of thousands of cases (training is between quadratic and cubic in \(n\)), and demands feature scaling; in exchange it is among the most accurate off-the-shelf classifiers for small, high-dimensional, noisy problems, which describes most biomedical datasets.</p></div>

<div class="exercise"><h4>Exercise 4</h4><p>In the kernel chunk of Section 7.3, increase the noise of the moons to 0.4 and refit. Which kernel degrades most? Then set <code>cost = 100</code> for the RBF machine and compare training and test accuracy. Explain the result using the slack variables of Section 7.2.</p></div>

## 8. Comparing the three families

### 8.1 An honest comparison on the full data

Every accuracy quoted so far was a single split or a cross-validation in which the hyper-parameters had been chosen by looking at the same folds. Supplement S1 explains why the second is optimistic: choosing \(k\) or \(C\) to maximise the cross-validated score and then reporting that score uses the validation data twice. The honest procedure is **nested cross-validation** — an inner cross-validation inside each training fold chooses the hyper-parameters, and the outer fold, never seen by the inner search, measures performance. The chunk runs it with stratified 5-fold outer folds for the three families, with the logistic regression of Unit 2 as reference, on all nine features, and reports mean ± standard deviation of accuracy and of the area under the ROC curve (AUC).



In [ ]:
# Nested 5-fold cross-validation of the three families (and logistic regression) on all nine features.
X9 <- bc[, 1:9]; yv <- bc$Class; set.seed(7); of <- sample(rep(1:5, length.out = nrow(bc)))
auc1 <- function(truth, score) as.numeric(pROC::auc(truth, score, levels = c("benign", "malignant"), direction = "<", quiet = TRUE))
mods <- c("Gaussian naive Bayes", "k-NN (k tuned)", "SVM linear (C tuned)", "SVM RBF (C,gamma tuned)", "logistic regression")
A <- Au <- matrix(NA, 5, 5, dimnames = list(NULL, mods))
for (f in 1:5) {
  tri <- of != f; Xtr <- X9[tri, ]; ytr <- yv[tri]; Xte <- X9[!tri, ]; yte <- yv[!tri]
  ct <- colMeans(Xtr); s <- apply(Xtr, 2, sd); Xtrs <- scale(Xtr, ct, s); Xtes <- scale(Xte, ct, s)
  nb <- naiveBayes(Xtr, ytr); A[f, 1] <- mean(predict(nb, Xte) == yte); Au[f, 1] <- auc1(yte, predict(nb, Xte, type = "raw")[, "malignant"])
  tk <- e1071::tune.knn(as.matrix(Xtrs), ytr, k = c(3, 5, 9, 15, 25))$best.parameters$k
  pk <- knn(Xtrs, Xtes, ytr, k = tk, prob = TRUE); pw <- attr(pk, "prob")
  A[f, 2] <- mean(pk == yte); Au[f, 2] <- auc1(yte, ifelse(pk == "malignant", pw, 1 - pw))
  sl <- e1071::tune.svm(Xtr, ytr, kernel = "linear", cost = c(0.1, 1, 10), probability = TRUE)$best.model
  A[f, 3] <- mean(predict(sl, Xte) == yte); Au[f, 3] <- auc1(yte, attr(predict(sl, Xte, probability = TRUE), "probabilities")[, "malignant"])
  sr <- e1071::tune.svm(Xtr, ytr, kernel = "radial", cost = c(1, 10, 100), gamma = c(0.03, 0.1, 0.3), probability = TRUE)$best.model
  A[f, 4] <- mean(predict(sr, Xte) == yte); Au[f, 4] <- auc1(yte, attr(predict(sr, Xte, probability = TRUE), "probabilities")[, "malignant"])
  lr <- suppressWarnings(glm(Class ~ ., data = bc[tri, ], family = binomial)); pl <- predict(lr, bc[!tri, ], type = "response")
  A[f, 5] <- mean(ifelse(pl > 0.5, "malignant", "benign") == yte); Au[f, 5] <- auc1(yte, pl)
}
cmp <- data.frame(accuracy = sprintf("%.3f +/- %.3f", colMeans(A), apply(A, 2, sd)),
                  AUC = sprintf("%.3f +/- %.3f", colMeans(Au), apply(Au, 2, sd)), row.names = mods)
print(cmp)



All five methods classify the Wisconsin data well — every AUC is close to 0.99 — but the accuracies are not all equal. All five methods reach accuracies around 0.96–0.97 that differ by less than the fold-to-fold standard deviation, so the table gives no grounds for preferring one of them. Naive Bayes, with no hyper-parameter to tune, is the fastest and has the lowest AUC of the five (though by a margin smaller than its standard deviation) — consistent with Section 4.5, it ranks cases a little less sharply because of its independence assumption on correlated features, yet classifies as accurately as the rest on this well-separated problem. The general lesson is typical of small, well-curated, low-dimensional problems: the methods are close, and the standard deviations across folds — of the order of 0.01–0.02 — are the resolution below which no claim should be made from a sample of 683 cases. Where the methods differ decisively is in what they offer besides accuracy, tabulated next.

### 8.2 When to use which

| | Naive Bayes | Bayesian network | k-NN | SVM |
|---|---|---|---|---|
| **Family** | generative, probabilistic | generative, probabilistic | memory-based | discriminative, geometric |
| **Main assumption** | features conditionally independent given the class | the dependence structure in the graph | near cases share labels; a meaningful distance exists | classes separable with a margin in the kernel's feature space |
| **Boundary shape** | quadratic (Gaussian), linear (equal variances) | depends on the model | arbitrary, local | linear or as flexible as the kernel |
| **Needs feature scaling** | no | no | **yes** | **yes** |
| **Hyper-parameters** | none (smoothing) | structure, priors | \(k\), metric, weighting | kernel, \(C\), \(\gamma\) |
| **Outputs probabilities** | yes, poorly calibrated | yes, and full posteriors | by vote fraction, coarse | no; Platt scaling needed |
| **Training cost** | one pass, \(O(np)\) | counting (known structure); search (unknown) | none | \(O(n^2)\)–\(O(n^3)\) |
| **Prediction cost** | \(O(p)\) | inference, exponential in the worst case | \(O(np)\) per query | \(O(\#\text{SV} \cdot p)\) |
| **Interpretability** | per-feature likelihoods | high: the graph is the explanation | by example | linear: coefficients; kernel: low |
| **Sample size** | works with very few cases | small if structure known | needs dense sampling | good with small \(n\), large \(p\) |
| **Handles missing values** | naturally (skip the feature) | naturally (marginalise) | needs imputation | needs imputation |
| **Typical biomedical use** | text, baselines, small-sample diagnosis | diagnosis with expert knowledge | imputation, similarity search | gene-expression and imaging, high-\(p\) |

Three rules of thumb follow. Use naive Bayes as the first baseline on any new problem and as a serious candidate when cases are scarce or features are counts. Use a Bayesian network when the structure of the problem is known and partially observed evidence must be reasoned about — when the question is not only "which class?" but "what is the probability of this finding given those?". Use k-NN when similarity is the natural notion (retrieval, imputation) and the data are dense in a low-dimensional representation; use a support vector machine when the data are high-dimensional, the sample is small, and accuracy matters more than probabilities or explanations. In every case, scale the features when the method is distance-based, fit the scaler inside the resampling, tune hyper-parameters by nested cross-validation, and report the spread across folds.

<div class="box bridge"><h4>Bridge to Units 4 and 6</h4><p>Unit 4 adds a fourth family, decision trees and random forests, which need no scaling, handle mixed data types natively and produce readable rules, and which on tabular biomedical data are often the strongest baseline. Unit 6 shows that a neural network with one hidden layer is also a kernel machine of sorts — it learns its own feature map \(\phi\) instead of fixing it in advance — and that the hinge and logistic losses of Figure 18 reappear as its output layer.</p></div>

## 9. Summary

You should now be able to explain, without notes:

- Bayes' theorem as the relation between the forward direction in which test characteristics are measured and the backward direction in which a clinician reasons, and why a positive result of an accurate test for a rare disease (prevalence 1 %, sensitivity and specificity 95 %) leaves a posterior of only about 16 %.
- The Bayes classifier, the Bayes error as the irreducible error of a problem, and how unequal costs replace the posterior threshold of 1/2 by \(\lambda_{\mathrm{FP}} / (\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}})\).
- The conditional-independence assumption of naive Bayes; the three steps (frequency table, likelihood table, posterior) of the categorical hand example; the Gaussian variant's parameters and log-posterior; Laplace smoothing and the zero-frequency problem; and the n-versus-(n − 1) variance convention that separates <code>GaussianNB</code> from <code>e1071</code>.
- Why naive Bayes classifies well although its assumption is false (ranking preserved while probabilities are distorted), and why its posteriors should not be trusted without calibration.
- A Bayesian network as a DAG plus conditional probability tables, the factorisation it encodes, inference by enumeration, explaining away, and the three d-separation structures (chain, fork, collider).
- Naive Bayes as the Bayesian network with one parent; parameter learning as counting; the existence and cost of structure learning.
- The k-NN rule as a lazy learner, its distance metrics, why feature scaling is mandatory, how \(k\) controls the bias–variance trade-off, what the curse of dimensionality does to "nearest", and why prediction rather than training is the expensive step.
- The maximum-margin hyperplane and support vectors; the soft margin and the meaning of \(C\); the kernel trick, the common kernels and the meaning of \(\gamma\); Platt scaling; the hinge loss and its relation to the logistic loss.
- How to compare classifier families honestly (nested cross-validation, mean ± sd across folds) and how to choose among them from their assumptions, scaling needs, costs and interpretability.

## 10. Self-assessment

<details><summary>1. A test has sensitivity 0.90 and specificity 0.90. Compute by hand the PPV when the prevalence is 2 %, and then when it is 50 %.</summary>At 2 %: \(P(D \mid T) = (0.9 \times 0.02) / (0.9 \times 0.02 + 0.1 \times 0.98) = 0.018 / 0.116 \approx 0.155\). At 50 %: \(0.45 / (0.45 + 0.05) = 0.90\). Only when the prior is 50 % does the PPV equal the sensitivity.</details>

<details><summary>2. In Figure 4 the minimum-risk threshold moved the boundary toward smaller clump-thickness values. If instead a false positive were five times as costly as a false negative, where would it move and what would the posterior threshold be?</summary>The threshold becomes \(\lambda_{\mathrm{FP}} / (\lambda_{\mathrm{FP}} + \lambda_{\mathrm{FN}}) = 5/6\): a case is called malignant only when the posterior exceeds 0.83, so the clump-thickness threshold moves to larger values and fewer cases are called malignant.</details>

<details><summary>3. In the weather example, compute by hand \(P(\text{Yes} \mid \text{Overcast}, \text{Hot}, \text{High}, \text{Windy} = \text{False})\) without smoothing, and explain what happens to the No class.</summary>Yes: \(9/14 \times 4/9 \times 2/9 \times 3/9 \times 6/9 \approx 0.0141\). No: \(5/14 \times 0/5 \times \dots = 0\). The normalised posterior of Yes is exactly 1 because the single zero likelihood \(P(\text{Overcast} \mid \text{No}) = 0\) annihilates the whole product for No — the zero-frequency problem that Laplace smoothing prevents.</details>

<details><summary>4. Gaussian naive Bayes with priors 0.5, benign mean (12, 2) and small variances, malignant mean (19, 9) and large variances. Without computing, say which class the point (12, 2) receives, and which class a point very far from both means receives, and why.</summary>(12, 2) is at the benign mean and far from the malignant one: benign. A point very far from both means is assigned to the class with the <em>larger variance</em> (malignant), because the quadratic penalty \((x - \mu)^2 / 2\sigma^2\) grows more slowly for the wide class; a narrow Gaussian makes distant points more surprising than a wide one does. (This is also why the six-case answer sits on the boundary and flips between the n and n − 1 variance conventions.)</details>

<details><summary>5. Naive Bayes reached about 0.96 accuracy with all nine features although cell size, cell shape and epithelial cell size are strongly correlated. Why did the redundancy not hurt the accuracy, and what did it hurt?</summary>Under zero-one loss only the ranking of the posteriors matters; counting the same evidence several times inflates both classes' log-likelihoods in a way that rarely changes which is larger (Domingos and Pazzani). What it hurts is calibration: the posteriors are pushed toward 0 and 1, so a large fraction of test cases receive a posterior above 0.99.</details>

<details><summary>6. In the chest-clinic network, is Smoking independent of X-ray? Is it independent of X-ray given Cancer? Is Cancer independent of Bronchitis given Dyspnoea? Justify each with the graph.</summary>No: the chain S → C → X is open, so S and X are dependent. Yes: conditioning on C blocks the chain. No: C → D ← B is a collider; conditioning on D opens the path, making C and B dependent (explaining away) — and the fork C ← S → B is also open unless S is observed.</details>

<details><summary>7. Count the free parameters of the full Asia network (8 binary nodes: Asia, Tuberculosis, Smoking, Cancer, Bronchitis, Either, X-ray, Dyspnoea) with arrows A → T, S → C, S → B, T → E, C → E, E → X, E → D, B → D.</summary>Asia 1, Tuberculosis 2, Smoking 1, Cancer 2, Bronchitis 2, Either 4 (two parents; deterministic in the original), X-ray 2, Dyspnoea 4: 18 parameters, against \(2^8 - 1 = 255\) for the unrestricted joint.</details>

<details><summary>8. In the <code>knn_cv</code> helper of Section 6.3, the scaler is fitted inside each fold (<code>scale(Xtr, ct, s)</code> with statistics from the training fold). Why is that necessary rather than scaling the whole matrix once before cross-validation?</summary>Fitting the scaler on the training fold only keeps the test fold's mean and variance out of the model; scaling the whole matrix beforehand would use the test cases' statistics — a (mild) form of data leakage that makes the estimate optimistic. Supplement S1 treats leakage in full.</details>

<details><summary>9. Figure 10 shows a training accuracy of exactly 1.000 at \(k = 1\). Would this still hold with distance weighting for any \(k\)? Would it hold with \(k = 1\) if the data contained two identical cases with different labels?</summary>With distance weights the training case itself is at distance 0 and receives (effectively) infinite weight, so training accuracy is 1.000 for every \(k\). With two identical cases of different labels, \(k = 1\) cannot classify both correctly, so training accuracy falls below 1.</details>

<details><summary>10. In Figure 13 the maximum-margin hyperplane was supported by very few cases. If one of the non-support cases were moved by a small amount, would the hyperplane change? If a support vector were moved?</summary>Moving a non-support case anywhere on its own side of the margin leaves the solution unchanged — it has zero weight \(\alpha_i\). Moving a support vector changes the hyperplane, because the solution is determined by the support vectors alone.</details>

<details><summary>11. Explain, in terms of the slack variables and the hinge loss, why an SVM with \(C = 100\) on noisy data has more support vectors than one with small \(C\), as in Figure 17.</summary>With a large \(C\) the optimiser is unwilling to pay for slack, so it narrows the margin and contorts the boundary (a large \(\gamma\) allows it) to reduce violations; many cases end up on or inside the narrow margin, where the hinge loss is active, and all of them become support vectors. With small \(C\) a wide margin is accepted, violations are cheap, and fewer cases determine the boundary.</details>

<details><summary>12. Section 8.1 reports accuracies for the support vector machines, k-NN and logistic regression that differ by less than the fold-to-fold standard deviation. A colleague proposes to pick the one with the highest mean and report it as "the best model for breast-cancer diagnosis". What is wrong, and what would you report?</summary>A difference smaller than the sampling variability is not evidence of a difference; the ranking would likely change with another split, and selecting the maximum of several noisy estimates is itself optimistically biased. Report all methods with mean ± sd, state that they are statistically indistinguishable on 683 cases, and choose by the secondary criteria of Section 8.2 (calibration, interpretability, cost), or run a paired test across folds if a decision must be defended.</details>

## Glossary

<dl class="gloss">
<dt>Bayes classifier / Bayes error</dt><dd>The rule that assigns each case to the class of highest posterior probability, optimal when the true distributions are known / its error rate, the irreducible error of the problem.</dd>
<dt>Bayesian network</dt><dd>A directed acyclic graph over random variables with a conditional probability table for each node given its parents; it encodes the factorisation of the joint distribution into local conditionals.</dd>
<dt>Calibration</dt><dd>Agreement between a classifier's predicted probabilities and the observed frequencies: among cases given probability 0.8, about 80 % should belong to the class.</dd>
<dt>Class-conditional density</dt><dd>\(p(\mathbf{x} \mid y = c)\), the distribution of the features among the cases of class \(c\); the object a generative classifier estimates.</dd>
<dt>Conditional independence</dt><dd>\(A\) and \(B\) are conditionally independent given \(C\) when \(P(A, B \mid C) = P(A \mid C)\,P(B \mid C)\); the assumption that makes naive Bayes and Bayesian networks tractable.</dd>
<dt>Curse of dimensionality</dt><dd>The phenomena by which geometric intuition fails as the number of features grows: volume concentrates in shells and distances between points become nearly equal.</dd>
<dt>d-separation</dt><dd>A graphical criterion for reading conditional independence off a Bayesian network: two nodes are independent given a set of observed nodes when every path between them is blocked (chains and forks blocked by observing the middle node; colliders blocked unless the collider or a descendant is observed).</dd>
<dt>Discriminative / generative</dt><dd>A classifier that models \(P(y \mid \mathbf{x})\) directly (logistic regression, SVM) / one that models \(p(\mathbf{x} \mid y)\) and \(P(y)\) and applies Bayes' theorem (naive Bayes, Bayesian networks).</dd>
<dt>Explaining away</dt><dd>The decrease in the posterior of one cause when another cause of the same observed effect is found to be present; a consequence of conditioning on a collider.</dd>
<dt>Hinge loss</dt><dd>\(\max(0, 1 - y f(\mathbf{x}))\), the loss minimised by the soft-margin SVM; zero for cases outside the margin, linear inside it.</dd>
<dt>Kernel / kernel trick</dt><dd>A function \(K(\mathbf{x}_i, \mathbf{x}_j)\) equal to the inner product of the two points in some feature space / the substitution of such a function for the inner product in an algorithm, making it non-linear without constructing the feature space.</dd>
<dt>k-nearest neighbours (k-NN)</dt><dd>The rule that classifies a case by the majority label (or predicts the mean target) among the \(k\) closest training cases under a chosen distance; a lazy learner.</dd>
<dt>Laplace (additive) smoothing</dt><dd>Adding a pseudo-count \(\alpha\) to every cell of a frequency table before estimating probabilities, so unseen feature values do not receive probability zero.</dd>
<dt>Lazy learner</dt><dd>A method that builds no model at training time and defers computation to prediction; k-NN is the archetype.</dd>
<dt>Margin</dt><dd>The perpendicular distance from a separating hyperplane to the closest training cases; the SVM maximises it.</dd>
<dt>Naive Bayes</dt><dd>The generative classifier that assumes the features are conditionally independent given the class; Gaussian, multinomial, Bernoulli and categorical variants differ in the one-dimensional model of each feature.</dd>
<dt>Platt scaling</dt><dd>Fitting a logistic function to a classifier's scores on held-out data to convert them into calibrated probabilities; the standard way to obtain probabilities from an SVM.</dd>
<dt>Posterior / prior / likelihood / evidence</dt><dd>The four terms of Bayes' theorem: \(P(\text{class} \mid \text{data})\) / \(P(\text{class})\) before the data / \(P(\text{data} \mid \text{class})\) / \(P(\text{data})\), the normalising constant.</dd>
<dt>Radial basis function (RBF) kernel</dt><dd>\(K = \exp(-\gamma \lVert \mathbf{x}_i - \mathbf{x}_j \rVert^2)\); the default SVM kernel, whose width is set by \(\gamma\).</dd>
<dt>Soft margin / C</dt><dd>The SVM formulation that allows margin violations measured by slack variables / the parameter that prices them, trading margin width against training errors.</dd>
<dt>Support vector</dt><dd>A training case that lies on or inside the margin (or is misclassified); the only cases that determine the SVM solution.</dd>
<dt>Validation curve</dt><dd>A plot of training and cross-validated performance against one hyper-parameter, used to choose its value and to see under- and overfitting.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Bayes T, Price R. An essay towards solving a problem in the doctrine of chances. <em>Philos Trans R Soc Lond</em>. 1763;53:370–418. <a href="https://doi.org/10.1098/rstl.1763.0053">doi:10.1098/rstl.1763.0053</a></li>
<li id="r2">Casscells W, Schoenberger A, Graboys TB. Interpretation by physicians of clinical laboratory results. <em>N Engl J Med</em>. 1978;299(18):999–1001. <a href="https://doi.org/10.1056/NEJM197811022991808">doi:10.1056/NEJM197811022991808</a></li>
<li id="r3">Gigerenzer G, Hoffrage U. How to improve Bayesian reasoning without instruction: frequency formats. <em>Psychol Rev</em>. 1995;102(4):684–704. <a href="https://doi.org/10.1037/0033-295X.102.4.684">doi:10.1037/0033-295X.102.4.684</a></li>
<li id="r4">Street WN, Wolberg WH, Mangasarian OL. Nuclear feature extraction for breast tumor diagnosis. <em>Proc SPIE</em>. 1993;1905:861–870. <a href="https://doi.org/10.1117/12.148698">doi:10.1117/12.148698</a></li>
<li id="r5">Domingos P, Pazzani M. On the optimality of the simple Bayesian classifier under zero-one loss. <em>Mach Learn</em>. 1997;29:103–130. <a href="https://doi.org/10.1023/A:1007413511361">doi:10.1023/A:1007413511361</a></li>
<li id="r6">Hand DJ, Yu K. Idiot's Bayes — not so stupid after all? <em>Int Stat Rev</em>. 2001;69(3):385–398. <a href="https://doi.org/10.1111/j.1751-5823.2001.tb00465.x">doi:10.1111/j.1751-5823.2001.tb00465.x</a></li>
<li id="r7">Ng AY, Jordan MI. On discriminative vs. generative classifiers: a comparison of logistic regression and naive Bayes. In: <em>Advances in Neural Information Processing Systems 14 (NIPS 2001)</em>. MIT Press; 2002:841–848. <a href="https://proceedings.neurips.cc/paper/2001/hash/7b7a53e239400a13bd6be6c91c4f6c4e-Abstract.html">proceedings.neurips.cc</a> (conference paper without DOI; official proceedings page)</li>
<li id="r8">Kononenko I. Machine learning for medical diagnosis: history, state of the art and perspective. <em>Artif Intell Med</em>. 2001;23(1):89–109. <a href="https://doi.org/10.1016/S0933-3657(01)00077-X">doi:10.1016/S0933-3657(01)00077-X</a></li>
<li id="r9">Pearl J. Fusion, propagation, and structuring in belief networks. <em>Artif Intell</em>. 1986;29(3):241–288. <a href="https://doi.org/10.1016/0004-3702(86)90072-X">doi:10.1016/0004-3702(86)90072-X</a></li>
<li id="r10">Pearl J. <em>Probabilistic Reasoning in Intelligent Systems: Networks of Plausible Inference</em>. Morgan Kaufmann; 1988. <a href="https://doi.org/10.1016/C2009-0-27609-4">doi:10.1016/C2009-0-27609-4</a></li>
<li id="r11">Lauritzen SL, Spiegelhalter DJ. Local computations with probabilities on graphical structures and their application to expert systems. <em>J R Stat Soc Ser B</em>. 1988;50(2):157–194. <a href="https://doi.org/10.1111/j.2517-6161.1988.tb01721.x">doi:10.1111/j.2517-6161.1988.tb01721.x</a></li>
<li id="r12">Quinlan JR. Induction of decision trees. <em>Mach Learn</em>. 1986;1:81–106. (Source of the fourteen-day weather dataset.) <a href="https://doi.org/10.1007/BF00116251">doi:10.1007/BF00116251</a></li>
<li id="r13">Heckerman D. A tutorial on learning with Bayesian networks. In: Jordan MI, ed. <em>Learning in Graphical Models</em>. Springer; 1998:301–354. <a href="https://doi.org/10.1007/978-94-011-5014-9_11">doi:10.1007/978-94-011-5014-9_11</a></li>
<li id="r14">Friedman N, Geiger D, Goldszmidt M. Bayesian network classifiers. <em>Mach Learn</em>. 1997;29:131–163. <a href="https://doi.org/10.1023/A:1007465528199">doi:10.1023/A:1007465528199</a></li>
<li id="r15">Lucas PJF, van der Gaag LC, Abu-Hanna A. Bayesian networks in biomedicine and health-care. <em>Artif Intell Med</em>. 2004;30(3):201–214. <a href="https://doi.org/10.1016/j.artmed.2003.11.001">doi:10.1016/j.artmed.2003.11.001</a></li>
<li id="r16">Fix E, Hodges JL. Discriminatory analysis. Nonparametric discrimination: consistency properties. USAF School of Aviation Medicine, Report 4; 1951. Reprinted in <em>Int Stat Rev</em>. 1989;57(3):238–247. <a href="https://doi.org/10.2307/1403797">doi:10.2307/1403797</a></li>
<li id="r17">Cover T, Hart P. Nearest neighbor pattern classification. <em>IEEE Trans Inf Theory</em>. 1967;13(1):21–27. <a href="https://doi.org/10.1109/TIT.1967.1053964">doi:10.1109/TIT.1967.1053964</a></li>
<li id="r18">Dudani SA. The distance-weighted k-nearest-neighbor rule. <em>IEEE Trans Syst Man Cybern</em>. 1976;SMC-6(4):325–327. <a href="https://doi.org/10.1109/TSMC.1976.5408784">doi:10.1109/TSMC.1976.5408784</a></li>
<li id="r19">Altman NS. An introduction to kernel and nearest-neighbor nonparametric regression. <em>Am Stat</em>. 1992;46(3):175–185. <a href="https://doi.org/10.1080/00031305.1992.10475879">doi:10.1080/00031305.1992.10475879</a></li>
<li id="r20">Bellman RE. <em>Adaptive Control Processes: A Guided Tour</em>. Princeton University Press; 1961. <a href="https://doi.org/10.1515/9781400874668">doi:10.1515/9781400874668</a></li>
<li id="r21">Beyer K, Goldstein J, Ramakrishnan R, Shaft U. When is "nearest neighbor" meaningful? In: <em>Database Theory — ICDT'99</em>. Lecture Notes in Computer Science, vol 1540. Springer; 1999:217–235. <a href="https://doi.org/10.1007/3-540-49257-7_15">doi:10.1007/3-540-49257-7_15</a></li>
<li id="r22">Bentley JL. Multidimensional binary search trees used for associative searching. <em>Commun ACM</em>. 1975;18(9):509–517. <a href="https://doi.org/10.1145/361002.361007">doi:10.1145/361002.361007</a></li>
<li id="r23">Troyanskaya O, Cantor M, Sherlock G, et al. Missing value estimation methods for DNA microarrays. <em>Bioinformatics</em>. 2001;17(6):520–525. <a href="https://doi.org/10.1093/bioinformatics/17.6.520">doi:10.1093/bioinformatics/17.6.520</a></li>
<li id="r24">Boser BE, Guyon IM, Vapnik VN. A training algorithm for optimal margin classifiers. In: <em>Proceedings of the Fifth Annual Workshop on Computational Learning Theory (COLT '92)</em>. ACM; 1992:144–152. <a href="https://doi.org/10.1145/130385.130401">doi:10.1145/130385.130401</a></li>
<li id="r25">Cortes C, Vapnik V. Support-vector networks. <em>Mach Learn</em>. 1995;20:273–297. <a href="https://doi.org/10.1007/BF00994018">doi:10.1007/BF00994018</a></li>
<li id="r26">Vapnik VN. <em>The Nature of Statistical Learning Theory</em>. Springer; 1995. <a href="https://doi.org/10.1007/978-1-4757-2440-0">doi:10.1007/978-1-4757-2440-0</a></li>
<li id="r27">Mercer J. Functions of positive and negative type, and their connection with the theory of integral equations. <em>Philos Trans R Soc Lond A</em>. 1909;209:415–446. <a href="https://doi.org/10.1098/rsta.1909.0016">doi:10.1098/rsta.1909.0016</a></li>
<li id="r28">Schölkopf B, Smola AJ. <em>Learning with Kernels: Support Vector Machines, Regularization, Optimization, and Beyond</em>. MIT Press; 2002. <a href="https://doi.org/10.7551/mitpress/4175.001.0001">doi:10.7551/mitpress/4175.001.0001</a></li>
<li id="r29">Platt JC. Probabilistic outputs for support vector machines and comparisons to regularized likelihood methods. In: Smola AJ, Bartlett P, Schölkopf B, Schuurmans D, eds. <em>Advances in Large Margin Classifiers</em>. MIT Press; 2000:61–74. <a href="https://doi.org/10.7551/mitpress/1113.001.0001">doi:10.7551/mitpress/1113.001.0001</a> (DOI of the edited volume; the chapter has no separate DOI that we could verify)</li>
<li id="r30">Lin H-T, Lin C-J, Weng RC. A note on Platt's probabilistic outputs for support vector machines. <em>Mach Learn</em>. 2007;68:267–276. <a href="https://doi.org/10.1007/s10994-007-5018-6">doi:10.1007/s10994-007-5018-6</a></li>
<li id="r31">Niculescu-Mizil A, Caruana R. Predicting good probabilities with supervised learning. In: <em>Proceedings of the 22nd International Conference on Machine Learning (ICML '05)</em>. ACM; 2005:625–632. <a href="https://doi.org/10.1145/1102351.1102430">doi:10.1145/1102351.1102430</a></li>
<li id="r32">Smola AJ, Schölkopf B. A tutorial on support vector regression. <em>Stat Comput</em>. 2004;14:199–222. <a href="https://doi.org/10.1023/B:STCO.0000035301.49549.88">doi:10.1023/B:STCO.0000035301.49549.88</a></li>
<li id="r33">Chang C-C, Lin C-J. LIBSVM: a library for support vector machines. <em>ACM Trans Intell Syst Technol</em>. 2011;2(3):27. <a href="https://doi.org/10.1145/1961189.1961199">doi:10.1145/1961189.1961199</a> (the "practical guide to SVM classification" by Hsu, Chang and Lin is distributed with the library; <code>e1071::svm</code> is its R interface)</li>
<li id="r34">Guyon I, Weston J, Barnhill S, Vapnik V. Gene selection for cancer classification using support vector machines. <em>Mach Learn</em>. 2002;46:389–422. <a href="https://doi.org/10.1023/A:1012487302797">doi:10.1023/A:1012487302797</a></li>
<li id="r35">Efron B, Hastie T, Johnstone I, Tibshirani R. Least angle regression. <em>Ann Stat</em>. 2004;32(2):407–499. (Source of the diabetes dataset used by the Python course; this R edition uses <code>BostonHousing</code> for regression.) <a href="https://doi.org/10.1214/009053604000000067">doi:10.1214/009053604000000067</a></li>
<li id="r36">R Core Team. R: A Language and Environment for Statistical Computing. R Foundation for Statistical Computing, Vienna. <a href="https://www.R-project.org/">https://www.R-project.org/</a> (software; no DOI)</li>
<li id="r37">Meyer D, Dimitriadou E, Hornik K, Weingessel A, Leisch F. e1071: Misc Functions of the Department of Statistics, Probability Theory Group, TU Wien. R package (naiveBayes, svm, tune). <a href="https://doi.org/10.32614/CRAN.package.e1071">doi:10.32614/CRAN.package.e1071</a></li>
<li id="r38">Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em>. 4th ed. Springer; 2002. (The <code>class</code> and <code>MASS</code> packages; <code>class::knn</code>.) <a href="https://doi.org/10.1007/978-0-387-21706-2">doi:10.1007/978-0-387-21706-2</a></li>
<li id="r39">James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — The R companion to the whole course; free PDF at statlearning.com. <a href="https://doi.org/10.1007/978-1-0716-1418-1">doi:10.1007/978-1-0716-1418-1</a></li>
<li id="r40">Leisch F, Dimitriadou E. mlbench: Machine Learning Benchmark Problems. R package (the Wisconsin breast-cancer, Pima and Boston data). <a href="https://doi.org/10.32614/CRAN.package.mlbench">doi:10.32614/CRAN.package.mlbench</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>James G, Witten D, Hastie T, Tibshirani R. <em>An Introduction to Statistical Learning with Applications in R</em>. 2nd ed. Springer; 2021. — Chapter 4 for Bayes classifiers and k-NN, Chapter 9 for SVMs; the most accessible treatment, and the R companion to this course.</li>
<li>Venables WN, Ripley BD. <em>Modern Applied Statistics with S</em> (MASS). 4th ed. Springer; 2002. — The reference for the R modelling functions used here, including <code>class::knn</code>.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. — Chapter 2.5 for the curse of dimensionality, 6.6.3 for naive Bayes, 12 for SVMs, 13 for k-NN. Free PDF.</li>
<li>Bishop CM. <em>Pattern Recognition and Machine Learning</em>. Springer; 2006. — 1.5 for decision theory, 2.5 for nearest neighbours, 7 for sparse kernel machines, 8 for graphical models. Free PDF.</li>
<li>Koller D, Friedman N. <em>Probabilistic Graphical Models: Principles and Techniques</em>. MIT Press; 2009. — The comprehensive reference on Bayesian networks, inference and learning.</li>
<li>Russell S, Norvig P. <em>Artificial Intelligence: A Modern Approach</em>. 4th ed. Pearson; 2021. — Chapters 12–13 for probability and Bayesian networks; Chapter 19 for k-NN and SVMs.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li>The <code>e1071</code>, <code>class</code>, <code>mlbench</code> and <code>pROC</code> package vignettes on CRAN — the documentation of every function used here.</li>
<li>LIBSVM and "A practical guide to support vector classification" — <a href="https://www.csie.ntu.edu.tw/~cjlin/libsvm/">csie.ntu.edu.tw/~cjlin/libsvm</a>; <code>e1071::svm</code> is the R interface.</li>
<li>The <code>bnlearn</code> R package and its repository of Bayesian networks, including the full Asia network with its published tables — <a href="https://www.bnlearn.com/bnrepository/">bnlearn.com/bnrepository</a>.</li>
<li>UCI Machine Learning Repository — <a href="https://archive.ics.uci.edu">archive.ics.uci.edu</a>; original source of the breast-cancer data.</li>
<li>Gigerenzer G. <em>Reckoning with Risk</em> (Penguin, 2002) and the Harding Center for Risk Literacy — <a href="https://www.hardingcenter.de">hardingcenter.de</a>; natural frequencies in medical communication.</li>
</ul>
